# QKD Eavesdropping Detection with Machine Learning (v2)

**Title:** ML-Based Eavesdropping Detection in BB84 (Fully Quantum) vs BKM07 (Semi-Quantum) Key Distribution Protocols

---

## What this notebook does

Quantum Key Distribution (QKD) lets two parties — traditionally called **Alice** and **Bob** — share a secret cryptographic key in a way that any eavesdropper (**Eve**) inevitably disturbs the channel and gets detected. But *how* do you detect that disturbance reliably, especially when the channel already has background noise?

This notebook answers that question using **machine learning**:

1. We **simulate** photon-by-photon runs of two QKD protocols (BB84 and BKM07) under various attack scenarios.
2. We **extract statistical features** from each simulated run — things like error rates, their variance over time, and spectral properties.
3. We **train and compare six ML classifiers** to distinguish "secure channel" from "Eve is present."
4. We **tune** three of those models with 5-fold cross-validation and plot ROC curves + feature importances.

### The two protocols

| Protocol | Type | Key idea |
|---|---|---|
| **BB84** (Bennett & Brassard, 1984) | Fully quantum | Alice sends single photons in one of two bases; Eve's intercept forces a random re-preparation, causing detectable errors |
| **BKM07** (Boyer–Kenigsberg–Mor, 2007) | Semi-quantum | Bob is "classical" — he can only measure in the Z-basis or reflect. Eve must attack both the forward and return legs to learn anything |

### Improvements in v2 over v1

| Area | Change |
|---|---|
| Feature engineering | Added `jump_energy`, `spectral_entropy`, `autocorr_lag1` derived from the per-window QBER time-series |
| Classifiers | Replaced NumPy hand-rolled models with scikit-learn; added SVM-RBF and XGBoost |
| Hyperparameter tuning | 5-fold stratified CV for Boosted Trees, SVM-RBF **and** Random Forest (previously only boosted was tuned) |
| Generalisation testing | Four distribution-shift splits (unseen attack intensity, unseen distance range, leave-one-cell-out) reporting per-class FNR, not one train/test noise split |
| Output | Feature importance plot in addition to ROC curves |

---
## Section 0 — Setup & Imports

We use:
- **NumPy / Matplotlib** for numerics and plotting
- **scikit-learn** for all ML models, pipelines, scaling, and evaluation
- **XGBoost** (optional) — the script gracefully falls back to scikit-learn's `HistGradientBoostingClassifier` if XGBoost is not installed

Run the cell below to install any missing packages, then import everything.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install scikit-learn ipykernel

^C


  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl (12.6 MB)
Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   ----- ---------------------------------- 1/7 [numpy]
   ----- ---------------------------------- 1/7

In [ ]:
# Uncomment if you need to install:
!pip install scikit-learn xgboost --break-system-packages
!pip install numpy --break-system-packages
!pip install matplotlib --break-system-packages
!pip install torch --break-system-packages

import numpy as np
import zlib
import matplotlib
matplotlib.use('Agg')          # use non-interactive backend (safe for notebooks too)
import matplotlib.pyplot as plt
import csv, os
from math import factorial
from scipy import stats
from scipy.optimize import brentq

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    IsolationForest,
    HistGradientBoostingClassifier,
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, average_precision_score
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

os.makedirs('data',  exist_ok=True)
os.makedirs('plots', exist_ok=True)

# ── P13: reproducibility -- one master seed, independent per-role streams ──
# Replaces the bare `np.random.seed(42)`. numpy's SeedSequence.spawn()
# guarantees statistically independent streams (plain np.random.seed does
# not). Streams are addressed by (role, index) so the SAME index gives the
# same channel realisation across attack classes (common random numbers --
# audit Sec. P13 rule 2 / P16's L-4 fix), which removes simulator noise from
# class comparisons and is what lets the leakage audit's nuisance-parameter
# check (L-4) pass honestly.
MASTER_SEED = 20260913


class SeedBook:
    """Deterministic, independent RNG streams, addressable by (role, index).

    Uses zlib.crc32 on the role string, NOT Python's builtin hash(): str
    hashing is salted by PYTHONHASHSEED, randomised per-process by default
    since Python 3.3 (security hardening against hash-flooding attacks),
    so hash(role) gives a DIFFERENT tag -- and therefore different actual
    random draws -- every time the notebook is run in a fresh process, even
    though it looks stable within any single run. crc32 is a fixed,
    unsalted function, so the same role always maps to the same stream
    across separate runs, which is the entire point of this class.
    """

    def __init__(self, master=MASTER_SEED):
        self.master = master
        self._roots = {}

    def _root(self, role):
        if role not in self._roots:
            tag = zlib.crc32(role.encode()) % (2 ** 31)
            self._roots[role] = np.random.SeedSequence([self.master, tag])
        return self._roots[role]

    def rng(self, role, index):
        return np.random.default_rng(self._root(role).spawn(index + 1)[index])


SEEDS = SeedBook()
N_REPEATS = 20  # >= 20 for a usable 95% CI on AUC-type metrics


def mean_ci(values, alpha=0.05):
    """Mean and normal-approximation 95% CI. Report this, not a bare number."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m = v.mean()
    se = v.std(ddof=1) / np.sqrt(len(v))
    return float(m), float(m - 1.96 * se), float(m + 1.96 * se)


print(f"XGBoost available : {HAS_XGB}")
print("All imports OK.")

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.3 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.4/9.3 MB 4.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.3 MB 5.1 MB/s eta 0:00:02
   --------------------- ------------------ 5.0/9.3 MB 5.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 6.8 MB/s eta 0:00:

---
## Section 1 — Simulating Qubit Physics

### Qubit state representation

A qubit (quantum bit) is represented as a 2-element complex vector:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1$$

We use two *bases*:

| Basis index | Name | $|0\rangle$-like state | $|1\rangle$-like state |
|---|---|---|---|
| 0 | **Z-basis** (rectilinear) | $|0\rangle = [1,\ 0]^T$ | $|1\rangle = [0,\ 1]^T$ |
| 1 | **X-basis** (diagonal) | $|+\rangle = \tfrac{1}{\sqrt{2}}[1,\ 1]^T$ | $|-\rangle = \tfrac{1}{\sqrt{2}}[1,\ -1]^T$ |

### Measurement (Born rule)

When Bob measures in basis $b$, the probability of getting outcome 0 is $|\langle b_0|\psi\rangle|^2$. The state then collapses to the measurement outcome.

### Noise model

`measure_qubit` exposes a single `noise_prob` knob (a depolarizing channel: with that probability the result is replaced by a uniformly random bit), used by BKM07's per-leg simulation. **BB84's actual noise is not a free parameter here** — Section 2 derives it from real fibre-link physics (loss, dark counts, detector misalignment) via `channel_model()`, and `noise_prob` for a given link is *read off* that model, not chosen directly.

> **Key insight:** Eve's intercept-resend attack introduces *extra* errors on top of the honest noise floor. The ML models learn to separate "noise-caused errors" from "Eve-caused errors" using the shape and temporal pattern of the error sequence, not just its average level.

In [ ]:
# ── Four fixed basis states ──────────────────────────────────────────────────
STATE_0    = np.array([1.0,  0.0], dtype=complex)
STATE_1    = np.array([0.0,  1.0], dtype=complex)
STATE_PLUS = np.array([1.0,  1.0], dtype=complex) / np.sqrt(2)
STATE_MINUS= np.array([1.0, -1.0], dtype=complex) / np.sqrt(2)


def prepare_state(bit, basis):
    '''Map a classical bit (0 or 1) into a quantum state vector.

    basis=0 -> Z-basis (|0> / |1>)
    basis=1 -> X-basis (|+> / |->)
    '''
    if basis == 0:
        return STATE_0.copy() if bit == 0 else STATE_1.copy()
    else:
        return STATE_PLUS.copy() if bit == 0 else STATE_MINUS.copy()


def measure_qubit(state, basis, noise_prob, rng=None):
    '''Measure a qubit in the given basis, subject to depolarizing noise.

    With probability `noise_prob` the channel corrupts the photon and we
    get a uniformly random bit (this is what drives QBER).
    Otherwise we use the Born-rule probability to decide the outcome.

    `rng`, if given, MUST be used for every random draw here: this function
    is called from simulate_bkm07_pulse, which threads a SEEDS-derived rng
    through everywhere else in that function. Drawing from the bare
    np.random module instead (as this used to do) silently bypasses that
    -- it makes BKM07's actual measurement outcomes (and therefore every
    BKM07 QBER feature) non-reproducible across runs even though the rest
    of the pipeline looks fully seeded. Defaults to a fresh unseeded
    generator only for standalone/demo use outside the main pipeline.

    Returns
    -------
    (measured_bit, collapsed_state)
    '''
    rng = rng or np.random.default_rng()
    # ── Noise: randomise the result ─────────────────────────────────────────
    if rng.random() < noise_prob:
        measured_bit = int(rng.integers(0, 2))
        return measured_bit, prepare_state(measured_bit, basis)

    # ── No noise: use quantum probability ───────────────────────────────────
    if basis == 0:
        prob_0 = float(np.abs(np.dot(STATE_0.conj(), state)) ** 2)
    else:
        prob_0 = float(np.abs(np.dot(STATE_PLUS.conj(), state)) ** 2)

    prob_0 = float(np.clip(prob_0, 0.0, 1.0))
    measured_bit = 0 if rng.random() < prob_0 else 1
    return measured_bit, prepare_state(measured_bit, basis)


# ── Quick sanity check ───────────────────────────────────────────────────────
# Prepare |+⟩, measure in Z-basis many times → should be ~50% each outcome
_demo_rng = np.random.default_rng(0)
results = [measure_qubit(STATE_PLUS.copy(), basis=0, noise_prob=0.0, rng=_demo_rng)[0]
           for _ in range(2000)]
print(f"Measuring |+⟩ in Z-basis 2000 times:")
print(f"  P(0) ≈ {results.count(0)/2000:.3f}  (expected 0.500)")
print(f"  P(1) ≈ {results.count(1)/2000:.3f}  (expected 0.500)")

Measuring |+⟩ in Z-basis 2000 times:
  P(0) ≈ 0.506  (expected 0.500)
  P(1) ≈ 0.494  (expected 0.500)


---
## Section 2 — Protocol Simulations & Attack Models

### BB84 pulse simulation

`simulate_bb84_decoy` models a full run of weak-coherent-pulse BB84, vectorised over N pulses at once (see Section 2c):

```
Alice → [photon in chosen basis, chosen intensity] → (Eve?) → Bob measures
```

**Attack modes:**

| `eve_mode` | What Eve does | Effect on QBER |
|---|---|---|
| `'none'` | No attack | Only channel noise |
| `'intercept_resend'` | Eve measures in a random basis, re-sends | Extra QBER: she guesses the wrong basis half the time, and half of those wrong-basis measurements collapse to the wrong bit |
| `'pns'` | Photon-Number-Splitting: Eve skims one photon from multi-photon pulses, forwards the rest losslessly | **Little to no extra QBER** — this is why PNS is dangerous, and why the decoy-state estimators (`y1_lower`, `e1_upper`, `gain_ratio_nu_mu`; Section 3) exist, not a raw photon-count feature Bob could never actually observe |

### BKM07 pulse simulation

BKM07 is a *semi-quantum* protocol: Alice is fully quantum, Bob only has classical capabilities (measure in Z or reflect unchanged).

```
Alice → [photon] → Bob (SIFT or CTRL) → [reflected photon] → Alice
```

- **SIFT:** Bob measures in Z, re-prepares, sends back. Alice's final measurement (also in her own preparation basis) establishes a key bit.
- **CTRL:** Bob reflects without measuring. Alice checks the return matches what she sent — disturbance here reveals Eve.

The channel is traversed **twice** (forward + return leg), so loss compounds as `eta_1way**2` and Eve must attack both legs to learn anything.

Rather than a free-parameter `noise_prob`, BB84/BKM07's honest QBER is derived from **named physical link quantities** (Ma, Qi, Zhao & Lo, Phys. Rev. A 72, 012326 (2005); GYS calibration from Gobby, Yuan & Shields, Appl. Phys. Lett. 84, 3762 (2004)):

- **Fibre attenuation** `alpha_db_km` — photon survival falls off as `10**(-alpha*d/10)` with distance.
- **Bob-side transmittance x detector efficiency** `eta_bob` — the *combined* internal loss and detector quantum efficiency at Bob's side (not raw detector efficiency alone).
- **Background/dark yield** `Y0` — detectors click at this rate even with no photon present; this is what makes honest QBER rise at long distance, as signal shrinks while dark clicks don't.
- **Detector misalignment error** `e_detector` — a distance-*independent* floor: even a lossless link has some probability of a wrong click from optical misalignment. This is what sets the honest-QBER floor at short range (~3.3% at 0km for the GYS parameters), not zero.
- **Background-count error rate** `e_0 = 0.5` — a dark click carries no information about the sent bit, so it's a coin flip by construction.

`channel_model(distance_km)` returns the resulting gain (`Q_mu`, fraction of pulses that produce a click) and QBER (`E_mu`) in closed form — Eqs. (10)-(11) of Ma et al. — and is the single source of truth every simulator (`simulate_bb84_decoy`, `simulate_bkm07_pulse`) and the calibration layer (Section 3) derives its noise from.

In [ ]:
# ── Physical channel model ───────────────────────────────────────────────
# Ma, Qi, Zhao & Lo, Phys. Rev. A 72, 012326 (2005), Sec. 2, Eqs. (4)-(11).
# Default constants are the GYS calibration (Gobby, Yuan & Shields,
# Appl. Phys. Lett. 84, 3762 (2004)) as tabulated in Ma et al. Table 1.
# Replaces physical_qber(), which mis-specified the detector error term as
# an absolute rather than a conditional probability and saturated at 50%
# QBER by 50 km (real fibre is flat ~3.3% out to 15 km, per GYS).
GYS = dict(alpha_db_km=0.21,   # fibre attenuation @1550 nm       [dB/km]
           eta_bob=0.045,      # Bob-side transmittance x detector efficiency
           Y0=1.7e-6,          # background/dark yield per pulse
           e_detector=0.033,   # optical misalignment error rate
           e_0=0.5,            # error rate of a background count
           f_ec=1.22)          # error-correction inefficiency

MU_SIGNAL, MU_DECOY, MU_VACUUM = 0.48, 0.05, 0.0   # Ma et al. Sec. 3.1 Eq. (12)
DECOY_PROBS = [0.70, 0.25, 0.05]
CHANNEL_DISTANCE_RANGE_KM = (0.0, 100.0)   # was (0, 15) -- see audit Sec. E.2


def channel_model(distance_km, mu=MU_SIGNAL, alpha_db_km=None, eta_bob=None,
                   Y0=None, e_detector=None, e_0=None):
    """Fibre-QKD channel: transmittance, yields, gain, QBER, single-photon terms.

    Returns a dict; `gain` is Q_mu and `qber` is E_mu in the notation of
    Ma et al. (2005).
    """
    alpha_db_km = GYS['alpha_db_km'] if alpha_db_km is None else alpha_db_km
    eta_bob = GYS['eta_bob'] if eta_bob is None else eta_bob
    Y0 = GYS['Y0'] if Y0 is None else Y0
    e_detector = GYS['e_detector'] if e_detector is None else e_detector
    e_0 = GYS['e_0'] if e_0 is None else e_0

    t_AB = 10.0 ** (-alpha_db_km * distance_km / 10.0)     # Eq. (5)
    eta = t_AB * eta_bob

    Q_mu = Y0 + 1.0 - np.exp(-eta * mu)                                        # Eq. (10)
    E_mu = (e_0 * Y0 + e_detector * (1.0 - np.exp(-eta * mu))) / Q_mu          # Eq. (11)

    Y1 = Y0 + eta - Y0 * eta                                                    # Eq. (7), i=1
    e1 = (e_0 * Y0 + e_detector * eta) / Y1                                     # Eq. (9), i=1

    return dict(distance_km=distance_km, transmittance=t_AB, eta=eta,
                gain=Q_mu, qber=E_mu, Y1=Y1, e1=e1,
                Y0=Y0, e_detector=e_detector, e_0=e_0, mu=mu)


# ── Validation against the GYS measurement ──────────────────────────────
print(f"{'L (km)':>8} {'QBER E_mu':>10} {'gain Q_mu':>12}")
for d in [0, 25, 50, 100, 122, 140]:
    c = channel_model(d)
    print(f"{d:>8.0f} {c['qber']:>10.4f} {c['gain']:>12.3e}")
print("GYS measured 8.9% QBER at 122 km (Appl. Phys. Lett. 84, 3762).")

  L (km)  QBER E_mu    gain Q_mu
       0     0.0330    2.137e-02
      25     0.0331    6.429e-03
      50     0.0334    1.925e-03
     100     0.0376    1.733e-04
     122     0.0460    6.092e-05
     140     0.0630    2.650e-05
GYS measured 8.9% QBER at 122 km (Appl. Phys. Lett. 84, 3762).


In [ ]:
# ── P3: Eve's PNS strategy, fixed ONCE from the signal intensity ───────────
# Brassard, Lutkenhaus, Mor & Sanders, PRL 85, 1330 (2000): Eve performs a
# QND photon-number measurement, blocks pulses with n < n_split, splits one
# photon from pulses with n >= n_split, and forwards the rest on a lossless
# line. She throttles the forwarded fraction t_fwd so Bob's observed gain
# matches the honest channel's. Eve cannot tell a signal pulse from a decoy
# pulse, so the SAME t_fwd is used for every intensity -- that asymmetry is
# exactly what the decoy-state method exploits.
def make_pns_strategy(distance_km, mu_signal=MU_SIGNAL, n_split=2, **chan):
    ch = channel_model(distance_km, mu=mu_signal, **chan)
    p_forwardable = sum(np.exp(-mu_signal) * mu_signal**k / factorial(k)
                         for k in range(n_split, 15))
    t_fwd = float(np.clip(ch['gain'] / max(p_forwardable, 1e-15), 0.0, 1.0))
    return dict(n_split=n_split, t_fwd=t_fwd, mu_signal=mu_signal, distance_km=distance_km)


# ── P4: Eve's activity as a stochastic process in TIME, not i.i.d. ─────────
# i.i.d. Eve has zero temporal structure, so the temporal features
# (variance, autocorrelation, spectral entropy) measure nothing. The
# profile is a modelling ASSUMPTION about Eve's operating mode (stated as
# such, not derived from a security proof).
def attack_schedule(n_pulses, intensity, profile='iid', rng=None,
                     mean_burst=2000, drift_period=None):
    rng = rng or np.random.default_rng()
    if intensity <= 0:
        return np.zeros(n_pulses, dtype=bool)
    if profile == 'iid':
        return rng.random(n_pulses) < intensity
    if profile == 'bursty':
        mask = np.zeros(n_pulses, dtype=bool)
        i, on = 0, (rng.random() < intensity)
        while i < n_pulses:
            L = min(max(1, int(rng.exponential(mean_burst))), n_pulses - i)
            mask[i:i + L] = on
            i += L
            on = rng.random() < intensity
        return mask
    if profile == 'drifting':
        T = drift_period or max(n_pulses // 4, 1)
        phase = rng.random() * 2 * np.pi
        env = intensity * (1.0 + 0.9 * np.sin(2 * np.pi * np.arange(n_pulses) / T + phase))
        return rng.random(n_pulses) < np.clip(env, 0.0, 1.0)
    raise ValueError(f"unknown profile {profile!r}")


# ── P5: vectorised decoy-state BB84 simulator ──────────────────────────────
# Replaces the per-pulse simulate_bb84_pulse() for dataset generation: it
# is exact i.i.d. sampling from the same joint distribution a per-pulse
# loop would draw one pulse at a time, just vectorised -- not an
# approximation. Required for N >= 1e6 pulses/run, which is what a real
# (lossy) channel needs before a window has enough sifted bits for the
# temporal features to mean anything (audit Sec. H.2).
def simulate_bb84_decoy(N, distance_km, eve_mode='none', eve_intensity=0.0,
                         profile='iid', pns_strategy=None, rng=None,
                         intensities=(MU_SIGNAL, MU_DECOY, MU_VACUUM),
                         probs=DECOY_PROBS, mean_burst=2000, **chan):
    """One BB84 run: weak coherent pulses, decoy intensities, real loss,
    dark counts, and a physically-implemented Eve.
    eve_mode in {'none','intercept_resend','pns','blocking','loss_manipulation'}
    """
    rng = rng or np.random.default_rng()
    mu_sig = intensities[0]
    ch = channel_model(distance_km, mu=mu_sig, **chan)
    eta, Y0, edet, e0 = ch['eta'], ch['Y0'], ch['e_detector'], ch['e_0']

    k = rng.choice(len(intensities), size=N, p=probs)
    mu_i = np.asarray(intensities)[k]
    n = rng.poisson(mu_i)                          # ALWAYS sampled (P2)
    bit_A = rng.integers(0, 2, N)
    bas_A = rng.integers(0, 2, N)
    bas_B = rng.integers(0, 2, N)

    active = attack_schedule(N, eve_intensity, profile, rng, mean_burst)   # P4
    eta_eff = np.full(N, float(eta))
    extra_err = np.zeros(N)

    # These two draws are made UNCONDITIONALLY (not just inside the
    # matching branch) so every eve_mode consumes the exact same amount of
    # the rng stream -- otherwise a zero-strength ('active' all False)
    # 'pns'/'intercept_resend' run would still land on a different point in
    # the random sequence than a 'none' run, which is an unnecessary,
    # avoidable confound for controls like the L-8 zero-strength audit
    # (P16) that compare classes at eve_intensity=0.
    bas_E = rng.integers(0, 2, N)
    pns_strat = pns_strategy or make_pns_strategy(distance_km, mu_sig, **chan)
    fwd = (n >= pns_strat['n_split']) & (rng.random(N) < pns_strat['t_fwd'])

    if eve_mode == 'intercept_resend':
        extra_err = np.where(active & (bas_E != bas_A), 0.5, 0.0)
    elif eve_mode == 'pns':
        eta_eff = np.where(active, np.where(fwd, 1.0, 0.0), eta)
    elif eve_mode == 'blocking':
        eta_eff = np.where(active, 0.0, eta)
    elif eve_mode == 'loss_manipulation':
        boost = 1.0 / max(1.0 - eve_intensity, 1e-6)
        eta_eff = np.where(active, 0.0, min(eta * boost, 1.0))

    p_sig = 1.0 - (1.0 - eta_eff) ** n
    sig_cl = rng.random(N) < p_sig
    dark = rng.random(N) < Y0
    click = sig_cl | dark

    p_err = np.where(sig_cl, np.clip(edet + extra_err * (1 - 2 * edet), 0, 1), e0)
    bit_B = np.where(rng.random(N) < p_err, 1 - bit_A, bit_A)
    sift = click & (bas_A == bas_B)

    Q, E, counts = {}, {}, {}
    for j, m in enumerate(intensities):
        sel = (k == j); s2 = sel & sift
        counts[m] = int(sel.sum())
        Q[m] = float(click[sel].mean()) if sel.any() else 0.0
        E[m] = float((bit_A[s2] != bit_B[s2]).mean()) if s2.any() else e0

    return dict(bit_A=bit_A, bit_B=bit_B, bas_A=bas_A, bas_B=bas_B, n=n, k=k,
                click=click, sift=sift, N=N, Q=Q, E=E, counts=counts,
                intensities=list(intensities), theory=ch)


# ── Decoy-state estimators (Ma et al. 2005, Eqs. 30 and 33) ────────────────
def decoy_estimate(Q_mu, E_mu, Q_nu, E_nu, Y0_obs, mu=MU_SIGNAL, nu=MU_DECOY, e0=0.5):
    """Vacuum+Weak decoy estimation of the single-photon yield and error rate."""
    Y1L = mu / (mu * nu - nu**2) * (
        Q_nu * np.exp(nu) - Q_mu * np.exp(mu) * nu**2 / mu**2
        - (mu**2 - nu**2) / mu**2 * Y0_obs)
    Y1L = max(float(Y1L), 0.0)
    Q1L = Y1L * mu * np.exp(-mu)
    e1U = ((E_nu * Q_nu * np.exp(nu) - e0 * Y0_obs) / (Y1L * nu)) if Y1L > 0 else 0.5
    return Y1L, Q1L, float(np.clip(e1U, 0.0, 0.5))


def secure_key_rate(Q_mu, E_mu, Q1, e1, f_ec=None, q=0.5):
    """GLLP + decoy asymptotic key rate; Lo, Ma & Chen PRL 94, 230504 (2005) Eq. 1.
    ASYMPTOTIC rate, used here as a channel-state FEATURE, not a security
    guarantee -- finite-key composable security needs additional terms
    (Tomamichel et al. 2012) not computed here."""
    f_ec = GYS['f_ec'] if f_ec is None else f_ec
    return max(q * (-Q_mu * f_ec * _binary_entropy(E_mu)
                     + Q1 * (1.0 - _binary_entropy(e1))), 0.0)


# ── P10: BKM07 protocol corrections ────────────────────────────────────────
# Boyer, Kenigsberg & Mor, PRL 99, 140501 (2007), Protocol 1:
#  * Alice prepares at random in |0>,|1>,|+>,|->.
#  * Classical Bob either SIFTs (Z-measure, re-prepare) or CTRLs (reflect).
#  * Alice measures the RETURN IN HER PREPARATION BASIS (not always Z, as
#    the previous version did).
#  * Alice-X + Bob-SIFT rounds are DISCARDED by the protocol; kept here only
#    as an auxiliary return-leg channel monitor (see collect_bkm07_features).
#  * Round trip means the channel is traversed TWICE: t^2, not t.
def simulate_bkm07_pulse(distance_km, eve_mode, eve_fwd, eve_ret, rng=None,
                          p_prep=None, **chan):
    rng = rng or np.random.default_rng()
    ch = channel_model(distance_km, **chan)
    eta_1way = ch['eta']
    p_prep = ch['e_detector'] if p_prep is None else p_prep
    p_err_leg = ch['e_detector']

    bit_A = int(rng.integers(0, 2))
    basis_A = int(rng.integers(0, 2))
    state = prepare_state(bit_A, basis_A)

    if rng.random() > eta_1way:
        return {'lost': True}

    if eve_mode != 'none' and rng.random() < eve_fwd:
        basis_Ef = int(rng.integers(0, 2))
        _, state = measure_qubit(state, basis_Ef, noise_prob=0.0, rng=rng)

    bob_mode = 'SIFT' if rng.random() < 0.5 else 'CTRL'
    bit_B = None
    if bob_mode == 'SIFT':
        bit_B, _ = measure_qubit(state, 0, 2 * p_err_leg, rng=rng)
        if rng.random() < p_prep:
            bit_B = 1 - bit_B
        state = prepare_state(bit_B, 0)

    if eve_mode != 'none' and rng.random() < eve_ret:
        basis_Er = int(rng.integers(0, 2))
        _, state = measure_qubit(state, basis_Er, noise_prob=0.0, rng=rng)

    if rng.random() > eta_1way:
        return {'lost': True}

    if bob_mode == 'CTRL':
        bit_A_final, _ = measure_qubit(state, basis_A, 2 * p_err_leg, rng=rng)
        round_type = 'CTRL_Z' if basis_A == 0 else 'CTRL_X'
    else:
        bit_A_final, _ = measure_qubit(state, 0, 2 * p_err_leg, rng=rng)
        round_type = 'SIFT_KEY' if basis_A == 0 else 'SIFT_MONITOR'

    return {'lost': False, 'bit_A': bit_A, 'basis_A': basis_A,
            'bob_mode': bob_mode, 'round_type': round_type,
            'bit_B': bit_B, 'bit_A_final': bit_A_final}


# ── Demo: QBER vs Eve intercept intensity (vectorised decoy simulator) ────
print("BB84 QBER vs Eve intercept intensity (distance=15km, N=5000):")
print(f"  {'Eve intensity':>14}  {'QBER':>6}")
for intensity in [0.0, 0.1, 0.2, 0.5, 1.0]:
    run = simulate_bb84_decoy(5000, 15.0, 'intercept_resend', intensity, rng=np.random.default_rng(0))
    print(f"  {intensity:>14.1f}  {run['E'][run['intensities'][0]]:>6.3f}")

BB84 QBER vs Eve intercept intensity (distance=15km, N=5000):
   Eve intensity    QBER
             0.0   0.000
             0.1   0.087
             0.2   0.087
             0.5   0.261
             1.0   0.435


### 2b — Photon-Number Statistics & PNS Vulnerability

A weak coherent pulse doesn't emit exactly one photon — the count follows a
Poisson distribution with mean `mu`. When `mu` rises, more pulses carry 2+
photons, which is exactly what a PNS attacker exploits: she can skim one
photon from a multi-photon pulse and forward the rest losslessly, without
Bob ever seeing a QBER change.

Bob cannot count photons directly, so `multi_rate` (the true photon-number
statistic) is **not observable** and is not a feature. What Bob *can*
observe is the **decoy-state trick**: Alice randomly varies the pulse
intensity (signal/decoy/vacuum) and Bob's gain/QBER at each intensity lets
her estimate a lower bound on the single-photon yield `Y1` and an upper
bound on its error rate `e1` (Ma et al. Eqs. 30, 33) — exactly the
`y1_lower`/`e1_upper`/`gain_ratio_nu_mu` features in Section 3.

In [ ]:

def simulate_photon_number_batch(mu, n_pulses, rng):
    return rng.poisson(mu, size=n_pulses)

def analyze_pns_vulnerability(mu_values, n_pulses=200_000, noise_prob=0.02, rng=None):
    rng = rng or np.random.default_rng(0)
    results = []
    for mu in mu_values:
        counts = simulate_photon_number_batch(mu, n_pulses, rng)
        vacuum_rate = (counts == 0).mean()
        single_rate = (counts == 1).mean()
        multi_rate  = (counts >= 2).mean()
        # QBER driven only by channel noise -- PNS itself adds none
        qber = (rng.random(n_pulses) < noise_prob).mean()
        results.append(dict(mu=mu, vacuum_rate=vacuum_rate,
                             single_rate=single_rate,
                             multi_rate=multi_rate, qber=qber))
    return results

mu_values = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
pns_results = analyze_pns_vulnerability(mu_values, rng=np.random.default_rng(0))
for r in pns_results:
    print(f"mu={r['mu']:.2f}  single={r['single_rate']:.3f}  "
          f"multi={r['multi_rate']:.3f}  QBER={r['qber']:.4f}")

mus     = [r['mu'] for r in pns_results]
multis  = [r['multi_rate'] for r in pns_results]
qbers   = [r['qber'] for r in pns_results]

fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(mus, multis, 'o-', color='#DC2626', label='multi-photon rate')
ax1.set_xlabel('mean photon number (mu)'); ax1.set_ylabel('multi-photon rate', color='#DC2626')
ax2 = ax1.twinx()
ax2.plot(mus, qbers, 's--', color='#2563EB', label='QBER')
ax2.set_ylabel('QBER', color='#2563EB')
plt.title('PNS vulnerability: multi-photon rate rises, QBER stays flat')
plt.tight_layout(); plt.savefig('plots/pns_vulnerability.png', dpi=150)
plt.show()


mu=0.05  single=0.047  multi=0.001  QBER=0.0194
mu=0.10  single=0.090  multi=0.005  QBER=0.0201
mu=0.15  single=0.129  multi=0.010  QBER=0.0202
mu=0.20  single=0.165  multi=0.018  QBER=0.0202
mu=0.30  single=0.221  multi=0.037  QBER=0.0199
mu=0.50  single=0.303  multi=0.089  QBER=0.0205


C:\Users\vigne\AppData\Local\Temp\ipykernel_5796\1582252983.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2c — The Monte Carlo approach used throughout feature extraction

Every feature-extraction call (`collect_bb84_features`, and every dataset
generated in Section 4) runs on `simulate_bb84_decoy`: a **vectorised**
Monte Carlo simulator that draws every random quantity (photon number,
bit, basis, click, error) for all N pulses as NumPy arrays in one shot,
rather than looping in Python pulse-by-pulse. That vectorisation is what
makes N=2,000,000 pulses/run — the scale Section 4 actually needs for the
windowed temporal features to be estimated above their binomial noise
floor (Section 3, and the N x window sensitivity study in Section 8b) —
run in a couple of seconds instead of minutes. The demo below runs it at
that scale and checks the empirical gain/QBER against `channel_model()`'s
closed-form prediction.

(BKM07's `simulate_bkm07_pulse` is a genuine *per-pulse* loop, not
vectorised — round-trip loss makes most pulses "lost" before they reach a
measurement, so `collect_bkm07_features` runs at a smaller N, set in
Section 4.)

In [ ]:
# ── The Monte Carlo approach actually used in feature extraction ──────────
# `simulate_bb84_decoy` (defined above) already IS the vectorized Monte
# Carlo simulator every feature-extraction call uses -- there is no
# separate "batch mode" needed. It draws N pulses' worth of every random
# quantity (photon number, bit, basis, click, error) as NumPy arrays in
# one shot rather than looping in Python, which is what makes N=2,000,000
# pulses/run (Section 4's dataset scale) practical. Demonstrating that
# here, at the scale collect_bb84_features() actually runs at:
import time

t0 = time.time()
demo_run = simulate_bb84_decoy(1_000_000, distance_km=25.0, eve_mode='none',
                                rng=np.random.default_rng(0))
elapsed = time.time() - t0
mu_sig = demo_run['intensities'][0]
print(f"1,000,000 pulses simulated in {elapsed:.2f}s "
      f"({1_000_000/max(elapsed, 1e-9)/1e6:.1f}M pulses/s)")
print(f"  signal-intensity gain Q_mu = {demo_run['Q'][mu_sig]:.4e}  "
      f"(theory: {demo_run['theory']['gain']:.4e})")
print(f"  signal-intensity QBER E_mu = {demo_run['E'][mu_sig]:.4f}  "
      f"(theory: {demo_run['theory']['qber']:.4f})")
print("Matches the closed-form channel_model() prediction: the simulator "
      "is drawing from that same distribution, just pulse-by-pulse.")

Total pulses: 1,000,000   Sifted: 499,517   QBER: 0.04681


---
## Section 3 — Feature Engineering

A single run of N pulses produces far too many individual events to feed directly into a classifier. Instead we compress each run into a **feature vector** — a small set of summary statistics that capture what matters.

### Why temporal features?

Average QBER alone can be ambiguous: channel noise and a mild attack can produce similar averages. But their **time profiles** differ:

- **Uniform noise** → QBER is roughly constant across time windows (low dispersion, low jump energy).
- **Intercept-resend attack** → Eve's interception can be bursty in time (Section 2's `profile` argument), creating correlations in the error sequence that a stationary honest channel doesn't have.
- **PNS attack** → QBER barely changes; only the decoy-state estimators see it (Section 2b).

### BB84 feature set (14 features, `BB84_FEATURE_NAMES`)

| Feature | What it measures |
|---|---|
| `qber_total` | Overall sifted error rate (signal intensity only) |
| `qber_z` / `qber_x` | Error rate within Z-basis / X-basis sifted rounds separately |
| `gain_mu` | Click probability at signal intensity — `Q_mu` in Section 2's channel model |
| `sifted_rate` | Fraction of signal pulses where Alice & Bob's bases matched |
| `qber_dispersion` | Ratio of observed per-window QBER variance to the binomial noise floor — ~1 for a stationary (honest or i.i.d.-Eve) process, >1 if the true error rate itself drifts in time |
| `jump_energy_norm` | Sum of squared consecutive per-window QBER differences, normalised by its value under stationarity — flags abrupt-onset attacks |
| `spectral_entropy` | Normalised Shannon entropy of the per-window QBER power spectrum (0 = structured/bursty, 1 = noise-like) |
| `autocorr_lag1` | Lag-1 autocorrelation of per-window QBER — positive if errors cluster in time |
| `y1_lower` | Decoy-state lower-bound estimate of the single-photon yield `Y1` (Ma et al. Eq. 30) |
| `e1_upper` | Decoy-state upper-bound estimate of the single-photon error rate `e1` (Eq. 33) |
| `r_secure` | GLLP+decoy asymptotic secure key rate, computed from `y1_lower`/`e1_upper` — used here as a channel-state *feature*, not a security certificate |
| `gain_ratio_nu_mu` | Ratio of decoy-to-signal gain (adjusted for intensity) — directly the quantity PNS distorts, since Eve cannot tell decoy pulses from signal pulses but the honest channel treats them identically |
| `h_qber` | Binary Shannon entropy `h(qber_total)` — nearly redundant with `qber_total` itself (r≈0.99), kept as a mild nonlinear transform |

### BKM07 feature set (14 features, `BKM_FEATURE_NAMES`)

BKM07 has separate forward and return legs, so we get more error channels:

| Feature | What it measures |
|---|---|
| `qber_key` | **True end-to-end key error**: `bit_A != bit_A_final` over SIFT_KEY rounds — the actual round-trip error a real deployment would report |
| `qber_zs` | Forward-leg + re-preparation error (Bob's SIFT measurement composed with his re-preparation flip; NOT the full round trip — see `qber_key`) |
| `qber_zsr` | Additional return-leg error on top of `qber_zs` (`bit_B != bit_A_final`) |
| `ret_monitor` | Return-leg error rate on X-basis SIFT rounds — protocol-discarded for key material, kept only as an auxiliary channel monitor |
| `qber_zc` / `qber_xc` | Z-basis / X-basis CTRL-round error rate (Alice checks her own reflected photon) |
| `asymmetry` | `|qber_zsr - qber_zs|` — signature of a return-leg-heavy attack |
| `ctrl_sift_ratio` | Average CTRL error rate relative to the SIFT forward-leg error rate |
| `xctrl_dispersion` | Ratio of per-window CTRL-X QBER variance to the binomial floor (BB84's `qber_dispersion`, applied to the CTRL-X monitor trace) |
| `jump_energy` | Temporal jump energy of the CTRL-X QBER trace |
| `spectral_entropy` | Spectral entropy of the CTRL-X QBER trace |
| `autocorr_lag1` | Lag-1 autocorrelation of the CTRL-X QBER trace |
| `h_ctrl` | Binary Shannon entropy of the average CTRL error rate |
| `sifted_rate` | Fraction of round trips that produced a SIFT_KEY round |

In [ ]:
# ─── Helper: binary entropy (Shannon, base-2) ────────────────────────────────
def _binary_entropy(p):
    '''H(p) = -p log2(p) - (1-p) log2(1-p).'''
    p = np.clip(p, 1e-10, 1.0 - 1e-10)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

# ─── Three temporal-shape features ───────────────────────────────────────────

def _jump_energy(x):
    '''Sum of squared consecutive differences.'''
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0.0
    return float(np.sum(np.diff(x) ** 2))


def _spectral_entropy(x, drop_dc=True):
    '''Normalised Shannon entropy of the power spectrum, in [0, 1].

    RELIABILITY NOTE: with W windows the rfft yields floor(W/2) usable bins
    after dropping DC. The plug-in entropy estimator is strongly negatively
    biased for small bin counts (Paninski, Neural Comput. 15, 1191 (2003)),
    so this feature is only meaningful for W >= 64; below that it is
    returned as NaN rather than silently producing a biased number.
    '''
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 8:
        return np.nan
    x = x - x.mean()
    spec = np.abs(np.fft.rfft(x)) ** 2
    if drop_dc:
        spec = spec[1:]
    tot = spec.sum()
    if tot <= 0:
        return 0.0
    p = spec / tot
    p = p[p > 0]
    if len(p) < 2:
        return np.nan
    return float(-np.sum(p * np.log2(p)) / np.log2(len(p)))


def _autocorr_lag1(x):
    '''Lag-1 autocorrelation. Near 0 = uncorrelated; positive = clustering.'''
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 3:
        return np.nan
    x0, x1 = x[:-1] - np.mean(x[:-1]), x[1:] - np.mean(x[1:])
    denom = np.sqrt(np.sum(x0 ** 2) * np.sum(x1 ** 2))
    return 0.0 if denom == 0 else float(np.sum(x0 * x1) / denom)


def _dispersion_index(window_errors, window_counts):
    '''Ratio of observed per-window QBER variance to the binomial floor.

    D = Var(Qhat) * nbar / (Qbar (1 - Qbar))

    D=1 : the QBER trace is consistent with a STATIONARY process observed
          through finite-sample noise (honest channel, or i.i.d. Eve).
    D>1 : over-dispersion -- the underlying error rate itself varies in
          time (bursty/drifting Eve, or a drifting channel).

    Replaces the raw `qber_variance`, which is a deterministic function of
    the mean QBER (measured ratio to this floor: 0.90-0.98) and therefore
    duplicates `qber_total` rather than adding information.
    '''
    we = np.asarray(window_errors, dtype=float)
    wc = np.asarray(window_counts, dtype=float)
    ok = wc > 0
    if ok.sum() < 4:
        return np.nan
    q = we[ok] / wc[ok]
    qb = we[ok].sum() / wc[ok].sum()
    nb = wc[ok].mean()
    floor = qb * (1 - qb) / nb
    return float(np.var(q) / floor) if floor > 0 else np.nan


def _jump_energy_norm(window_errors, window_counts):
    '''Sum of squared consecutive QBER differences, normalised by the value
    expected for a white (stationary) sequence: 2*(W-1)*Qbar(1-Qbar)/nbar.
    ~1 under stationarity, >1 for abrupt transitions.'''
    we = np.asarray(window_errors, dtype=float); wc = np.asarray(window_counts, dtype=float)
    ok = wc > 0
    if ok.sum() < 3:
        return np.nan
    q = we[ok] / wc[ok]
    qb = we[ok].sum() / wc[ok].sum(); nb = wc[ok].mean()
    expected = 2 * (len(q) - 1) * qb * (1 - qb) / nb
    return float(np.sum(np.diff(q) ** 2) / expected) if expected > 0 else np.nan


# Used by the BKM07 calibration empirical check (Section 3's matched-QBER
# layer) to report a statistically correct binomial CI, rather than a
# normal approximation, on the measured qber_key.
def qber_ci(errors, total, alpha=0.05):
    '''Exact (Clopper-Pearson) confidence interval for an observed QBER.'''
    if total == 0:
        return np.nan, 0.0, 1.0
    q = errors / total
    lo = stats.beta.ppf(alpha / 2, errors, total - errors + 1) if errors > 0 else 0.0
    hi = stats.beta.ppf(1 - alpha / 2, errors + 1, total - errors) if errors < total else 1.0
    return q, float(lo), float(hi)


print("Temporal-feature helpers defined (P6: fixed spectral entropy, added dispersion index).") #bechmark: 2024-06-13

Temporal-feature helpers defined (P6: fixed spectral entropy, added dispersion index).


---
## Section 3b — E91 Feature Engineering

### Why CHSH/detector features, not just QBER?

E91 gives you a second aggregate statistic BB84/BKM07 don't have — the CHSH value `S` — but averaging either `S` or QBER over a whole run can still miss a bursty or basis-anisotropic attack:

- **Intercept-resend** → `S` collapses toward the classical bound *and* QBER rises together, in a way consistent with the honest depolarisation curve (see `s_qber_residual` below) — this is what makes it hard to separate from ordinary channel noise using `S`/QBER aggregates alone, and why the temporal features matter here too.
- **Entangling-ancilla probe** → costs disproportionately *more* QBER than it costs `S`, because it dephases anisotropically rather than depolarising the state — `s_qber_residual` is built specifically to catch this.
- **Asymmetric arm-loss manipulation** → looks similar to honest visibility reduction at the aggregate level; separable mainly via its distinct effect on `sifted_rate` and its temporal profile.

### E91 feature set (12 features, produced by `extract_e91_features`)

| Feature | Formula / idea |
|---|---|
| `qber_key` | Error rate in matching-angle, key-generating rounds — `(a2,b1)`, `(a3,b2)` |
| `chsh_S` | Measured CHSH statistic from the four non-matching test-angle combinations |
| `s_deviation` | $|2\sqrt{2} - |\text{chsh\_S}||$ — distance below the Tsirelson bound |
| `s_qber_residual` | Deviation of measured `|S|` from the honest-depolarisation curve `2*sqrt(2)*(1-2*qber_key)` — near 0 for honest noise *and* basis-averaged intercept-resend, but distinctly nonzero for the basis-anisotropic ancilla attack (Section 13) |
| `per_pair_corr_spread` | max − min across the four individual `E(a_i,b_j)` correlations feeding `S` — an attack biased toward one angle setting hides in the aggregate but stands out here |
| `jump_energy` | Sum of squared consecutive per-window `chsh_S` differences — penalises sudden-onset attacks |
| `spectral_entropy` | Normalised Shannon entropy of the per-window QBER power spectrum (0 = structured/bursty, 1 = noise-like) |
| `autocorr_lag1` | Lag-1 autocorrelation of per-window `qber_key` — positive if errors cluster in time |
| `h_qber_key` | Binary Shannon entropy `h(qber_key)` |
| `sifted_rate` | Fraction of pairs landing on a matching (key-generating) angle combination |
| `attack_duty_cycle` | Fraction of the run during which Eve was actively attacking (0 for honest runs) — a channel-state feature, not something a real deployment could directly read off, but useful here for understanding what the classifier is learning |
| `V` | The honest-channel Werner-state visibility drawn for this run — included as a feature so the model can, in principle, separate visibility-driven degradation from an actual attack |

Note what's **not** here versus the original design: `g2_0`, `detection_efficiency`, `detector_curve_residual`, and `photocurrent_baseline` (source/detector-hardware statistics for a PNS/blinding analogue) were removed — they were sampled directly from label-conditioned distributions rather than genuinely simulated, which made them trivially separable by construction (Section 13's note). Real detector-side attacks need a real detector model to characterise honestly, which is out of scope here (Section 13's scope note).

In [ ]:
"""
E91 Feature Engineering -- genuine two-qubit density-matrix version
=====================================================================
P7: replaces the analytic-correlation table lookup (which could not
    represent decoherence, Eve's ancilla, or asymmetric loss because it
    never instantiated a quantum state) with an actual 4x4 density matrix
    and the Born rule. The honest channel is a WERNER state with visibility
    V drawn from a realistic range -- so, unlike before, honest degradation
    is no longer exactly zero and "any deviation = Eve" is no longer true
    by construction.
P8: Eve's attacks are genuine CPTP maps (measure-and-resend, entangling
    ancilla probe), basis-averaged rather than resampled per CHSH
    correlator (the previous version's sin^2/cos^2 convention mismatch and
    per-correlator resampling gave a WRONG |S| for intercept-resend: 2.77
    instead of the textbook 1.39-1.41).
P9: the four features sampled from label-conditioned distributions
    (g2_0, detection_efficiency, detector_curve_residual,
    photocurrent_baseline) are REMOVED -- they were not simulated
    measurements, they were draws from N(mu_A, sd_A) vs N(mu_B, sd_B) with
    disjoint support chosen by the ground-truth label, which is why they
    hit AUC~1.0. Replaced by `s_qber_residual`, a physics-derived feature:
    honest depolarisation obeys |S| = 2*sqrt(2)*(1-2Q), so the deviation
    from that curve is a genuine anisotropy diagnostic (measured ~0 for
    honest noise AND for intercept-resend, but +0.16 to +0.28 for an
    entangling-ancilla attack -- see the demo at the bottom of this cell).
`detector_blind`/`pns` are dropped from E91's eve_modes (they were
label-conditioned constructs, not simulated attacks) and replaced with
`ancilla` (entangling probe) and `loss_manipulation` (asymmetric arm
loss), per the recommended E91 attack set (audit Sec. D.3).
"""

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Protocol constants (Ekert 1991 optimal angles, polarisation convention:
# E(theta_a,theta_b) = -cos(2*(theta_a-theta_b)))
# ---------------------------------------------------------------------
ALICE_ANGLES = {"a1": 0.0, "a2": 22.5, "a3": 45.0}
BOB_ANGLES = {"b1": 22.5, "b2": 45.0, "b3": 67.5}
KEY_PAIRS = [("a2", "b1"), ("a3", "b2")]
CHSH_PAIRS = [("a1", "b1"), ("a1", "b3"), ("a3", "b1"), ("a3", "b3")]
CHSH_SIGNS = [+1, -1, +1, +1]
TSIRELSON_BOUND = 2 * np.sqrt(2)

# ---------------------------------------------------------------------
# P7: genuine two-qubit state, Born-rule sampling
# ---------------------------------------------------------------------
I2 = np.eye(2, dtype=complex)
SX = np.array([[0, 1], [1, 0]], dtype=complex)
SZ = np.array([[1, 0], [0, -1]], dtype=complex)
PSI_MINUS = np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2)
RHO_SINGLET = np.outer(PSI_MINUS, PSI_MINUS.conj())


def polarisation_observable(theta_deg):
    """A(theta) = cos(2 theta) sigma_z + sin(2 theta) sigma_x.
    For the singlet this gives <A(a) (x) A(b)> = -cos(2(a-b)), Ekert's convention."""
    t = np.radians(theta_deg)
    return np.cos(2 * t) * SZ + np.sin(2 * t) * SX


def projectors(theta_deg):
    A = polarisation_observable(theta_deg)
    return {+1: (I2 + A) / 2, -1: (I2 - A) / 2}


def werner_state(V, rho=RHO_SINGLET):
    """rho_W = V |Psi-><Psi-| + (1-V) I/4. Werner, PRA 40, 4277 (1989).
    Gives |S|_max = 2 sqrt(2) V and key-basis QBER = (1-V)/2."""
    return V * rho + (1.0 - V) * np.eye(4, dtype=complex) / 4.0


def joint_probs(rho, a_deg, b_deg):
    """Born rule: P(x,y|a,b) = Tr[rho (Pi_a^x (x) Pi_b^y)]."""
    PA, PB = projectors(a_deg), projectors(b_deg)
    out = {}
    for x in (+1, -1):
        for y in (+1, -1):
            out[(x, y)] = float(np.real(np.trace(rho @ np.kron(PA[x], PB[y]))))
    s = sum(out.values())
    return {kk: max(v, 0.0) / s for kk, v in out.items()}


def sample_e91_channel(rng):
    """Honest-link visibility. V=1 is the ideal singlet; the honest range
    below gives a genuine, non-zero noise floor (audit Defect 7): any run
    with V < 1 has real, non-adversarial degradation to compare Eve against."""
    return float(rng.uniform(0.85, 0.99))


# ---------------------------------------------------------------------
# P8: Eve as genuine CPTP maps (not per-correlator resampled table lookups)
# ---------------------------------------------------------------------
EVE_BASES = (0.0, 22.5, 45.0, 67.5)


def eve_measure_resend_bob(rho, e_deg):
    """Eve intercepts Bob's photon, measures at e_deg, resends the eigenstate
    (measure-and-reprepare CPTP map on the pair state)."""
    PB = projectors(e_deg)
    out = np.zeros((4, 4), dtype=complex)
    for y in (+1, -1):
        K = np.kron(I2, PB[y])
        red = K @ rho @ K
        pA = np.zeros((2, 2), dtype=complex)   # partial trace over Bob
        for i in range(2):
            for j in range(2):
                pA[i, j] = red[2 * i, 2 * j] + red[2 * i + 1, 2 * j + 1]
        out += np.kron(pA, PB[y])
    return out / np.real(np.trace(out))


def eve_ir_channel(rho, bases=EVE_BASES):
    """Eve's basis is random PER PULSE, so the physically correct object is
    the BASIS-AVERAGED map, identical for every Alice/Bob setting pair.
    (Sampling a fresh basis per CHSH correlator, as the previous version
    did, mixes four inconsistent attacks and can push |S| above what any
    single attack could produce -- measured 2.77 instead of the correct
    ~1.39-1.41 for 100% intercept-resend.)"""
    return sum(eve_measure_resend_bob(rho, e) for e in bases) / len(bases)


def eve_ancilla_bob(rho, lam, basis_deg=0.0):
    """Entangling probe: Eve couples an ancilla to Bob's qubit and keeps it.
    Tracing out the ancilla leaves basis-dependent (ANISOTROPIC) dephasing
    of strength lam -- unlike depolarisation, this costs more QBER than it
    costs CHSH, which is what makes it separable from honest noise via
    `s_qber_residual` below. Simplified individual-attack version of
    Fuchs et al., PRA 56, 1163 (1997)."""
    A = polarisation_observable(basis_deg)
    K0 = np.sqrt(1.0 - lam / 2.0) * I2
    K1 = np.sqrt(lam / 2.0) * A
    out = np.zeros((4, 4), dtype=complex)
    for K in (K0, K1):
        M = np.kron(I2, K)
        out += M @ rho @ M.conj().T
    return out / np.real(np.trace(out))


def eve_loss_manipulation(rho, delta=0.3):
    """Asymmetric arm loss: Eve attenuates Bob's arm more than Alice's.
    Modelled as an isotropic visibility reduction on top of the honest
    channel (the correlation degrades, but the mechanism -- differential
    loss -- is a channel-level attack, not a measurement)."""
    return (1.0 - delta) * rho + delta * np.eye(4, dtype=complex) / 4.0


# (delta is scaled down from eve_intensity below so this does not
# numerically coincide with eve_ir_channel at eve_intensity=0.5)


def run_e91(n_pairs, V=0.95, eve_mode='none', eve_intensity=0.0, lam=0.3,
            profile='iid', rng=None, mean_burst=2000):
    """One E91 run. Outcomes are drawn from the exact Born-rule joint
    distribution for each (setting pair, channel condition) -- there are
    only 3x3 settings x 2 channel conditions, so 18 probability vectors are
    computed per run and sampled from. Multinomial sampling from the Born
    probabilities is identically distributed to shot-by-shot statevector
    simulation; it is exact, not an approximation (measured: 200,000 pairs
    in 0.07s -- see the P20 Qiskit cross-check in the appendix, if run)."""
    rng = rng or np.random.default_rng()
    rho0 = werner_state(V)
    attacked = attack_schedule(n_pairs, eve_intensity, profile, rng, mean_burst)

    a_keys = list(ALICE_ANGLES.keys())
    b_keys = list(BOB_ANGLES.keys())
    ak = rng.choice(a_keys, size=n_pairs)
    bk = rng.choice(b_keys, size=n_pairs)
    ra = np.zeros(n_pairs, dtype=int)
    rb = np.zeros(n_pairs, dtype=int)

    if eve_mode == 'intercept_resend':
        rho_att = eve_ir_channel(rho0)
    elif eve_mode == 'ancilla':
        rho_att = eve_ancilla_bob(rho0, lam)
    elif eve_mode == 'loss_manipulation':
        rho_att = eve_loss_manipulation(rho0, delta=min(0.35, 0.5 * eve_intensity))
    else:
        rho_att = rho0

    for a_name, a_deg in ALICE_ANGLES.items():
        for b_name, b_deg in BOB_ANGLES.items():
            for att in (False, True):
                m = (ak == a_name) & (bk == b_name) & (attacked == att)
                cnt = int(m.sum())
                if cnt == 0:
                    continue
                p = joint_probs(rho_att if att else rho0, a_deg, b_deg)
                keys = list(p)
                idx = rng.choice(len(keys), size=cnt, p=[p[kk] for kk in keys])
                outs = np.array(keys)[idx]
                ra[m], rb[m] = outs[:, 0], outs[:, 1]

    return ak, bk, ra, rb


# ---------------------------------------------------------------------
# Windowed protocol statistics (unchanged -- these already operate on the
# named-setting / +-1-outcome interface run_e91() above still provides)
# ---------------------------------------------------------------------
def window_qber(a_choice, b_choice, r_a, r_b, start, end):
    key_a, key_b = [], []
    for ac, bc in KEY_PAIRS:
        m = (a_choice[start:end] == ac) & (b_choice[start:end] == bc)
        key_a.extend(r_a[start:end][m])
        key_b.extend(-r_b[start:end][m])  # flip: singlet -> anti-correlated
    if len(key_a) == 0:
        return np.nan
    key_a, key_b = np.array(key_a), np.array(key_b)
    return np.mean(key_a != key_b)


def window_chsh(a_choice, b_choice, r_a, r_b, start, end):
    corrs = []
    for ac, bc in CHSH_PAIRS:
        m = (a_choice[start:end] == ac) & (b_choice[start:end] == bc)
        if m.sum() == 0:
            return np.nan
        corrs.append(np.mean(r_a[start:end][m] * r_b[start:end][m]))
    return sum(s * c for s, c in zip(CHSH_SIGNS, corrs))


def windowed_traces(a_choice, b_choice, r_a, r_b, n_windows):
    n = len(a_choice)
    edges = np.linspace(0, n, n_windows + 1, dtype=int)
    qber_trace, chsh_trace = [], []
    for start, end in zip(edges[:-1], edges[1:]):
        qber_trace.append(window_qber(a_choice, b_choice, r_a, r_b, start, end))
        chsh_trace.append(window_chsh(a_choice, b_choice, r_a, r_b, start, end))
    return np.array(qber_trace), np.array(chsh_trace)


def spectral_entropy(x):
    return _spectral_entropy(x)


def autocorr_lag1(x):
    return _autocorr_lag1(x)


def jump_energy(x):
    return _jump_energy(x)


def binary_entropy(p):
    return _binary_entropy(p)


# ---------------------------------------------------------------------
# Full feature extraction for one run (P9: 4 leaking features removed,
# s_qber_residual added)
# ---------------------------------------------------------------------
def extract_e91_features(n_pulses=2000, eve_mode="none", n_windows=20, rng=None,
                          eve_intensity=0.0, lam=0.3, profile='iid', V=None):
    rng = rng or np.random.default_rng()
    V = sample_e91_channel(rng) if V is None else V

    a_choice, b_choice, r_a, r_b = run_e91(n_pulses, V=V, eve_mode=eve_mode,
                                            eve_intensity=eve_intensity, lam=lam,
                                            profile=profile, rng=rng)
    qber_trace, chsh_trace = windowed_traces(a_choice, b_choice, r_a, r_b, n_windows)

    qber_key = window_qber(a_choice, b_choice, r_a, r_b, 0, n_pulses)
    chsh_s = window_chsh(a_choice, b_choice, r_a, r_b, 0, n_pulses)

    corrs = []
    for ac, bc in CHSH_PAIRS:
        m = (a_choice == ac) & (b_choice == bc)
        corrs.append(np.mean(r_a[m] * r_b[m]) if m.sum() else np.nan)
    per_pair_corr_spread = float(np.nanmax(corrs) - np.nanmin(corrs))

    sifted_rate = float(np.mean([
        ((a_choice == ac) & (b_choice == bc)).sum() for ac, bc in KEY_PAIRS
    ]) / n_pulses * len(KEY_PAIRS))

    # P9: s_qber_residual -- physics-derived anisotropy detector.
    # Honest depolarisation obeys |S| = 2*sqrt(2)*(1-2Q); deviation from
    # that curve indicates an ANISOTROPIC disturbance (e.g. an entangling
    # probe), as opposed to ordinary white noise. Measured: ~0 for honest
    # channels AND for basis-averaged intercept-resend; +0.16 to +0.28 for
    # ancilla attacks.
    if not np.isnan(chsh_s) and not np.isnan(qber_key):
        s_pred = TSIRELSON_BOUND * (1 - 2 * qber_key)
        s_qber_residual = float(abs(chsh_s) - s_pred)
    else:
        s_qber_residual = np.nan

    features = {
        "qber_key": qber_key,
        "chsh_S": chsh_s,
        "s_deviation": abs(TSIRELSON_BOUND - abs(chsh_s)) if not np.isnan(chsh_s) else np.nan,
        "s_qber_residual": s_qber_residual,
        "per_pair_corr_spread": per_pair_corr_spread,
        "jump_energy": float(jump_energy(chsh_trace)),
        "spectral_entropy": float(spectral_entropy(qber_trace)),
        "autocorr_lag1": float(autocorr_lag1(qber_trace)),
        "h_qber_key": float(binary_entropy(qber_key)) if not np.isnan(qber_key) else np.nan,
        "sifted_rate": sifted_rate,
        "attack_duty_cycle": eve_intensity if eve_mode != 'none' else 0.0,
        "V": V,
    }
    return features


# ---------------------------------------------------------------------
# Dataset generator (labeled, for classifier training)
# eve_modes updated per P9 / audit Sec. D.3: drop detector_blind/pns
# (label-conditioned constructs), add ancilla (entangling probe) and
# loss_manipulation (asymmetric arm loss).
# ---------------------------------------------------------------------
def generate_e91_dataset(n_runs_per_mode=100, n_pulses=2000, n_windows=20,
                          eve_modes=("none", "intercept_resend", "ancilla", "loss_manipulation"),
                          seed_role='e91_run'):
    rows = []
    for mode in eve_modes:
        for i in range(n_runs_per_mode):
            rng = SEEDS.rng(seed_role, i) if mode == "none" else SEEDS.rng(seed_role + '_' + mode, i)
            intensity = float(rng.uniform(0.1, 0.9)) if mode != "none" else 0.0
            profile = 'bursty' if (mode == 'intercept_resend' and rng.random() < 0.5) else 'iid'
            feats = extract_e91_features(n_pulses=n_pulses, eve_mode=mode,
                                          n_windows=n_windows, rng=rng,
                                          eve_intensity=intensity, profile=profile)
            feats["label"] = mode
            rows.append(feats)
    return pd.DataFrame(rows)


if __name__ == "__main__":
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 20)

    print("Physics validation -- Werner-state E91 (200,000 pairs per row):")
    print(f"{'V':>6} {'|S| measured':>14} {'2sqrt2*V':>10} {'QBER measured':>15} {'(1-V)/2':>10}")
    for V in (1.00, 0.95, 0.90, 0.85):
        _rng = np.random.default_rng(7)
        _a, _b, _ra, _rb = run_e91(200_000, V=V, eve_mode='none', rng=_rng)
        _s = window_chsh(_a, _b, _ra, _rb, 0, len(_a))
        _q = window_qber(_a, _b, _ra, _rb, 0, len(_a))
        print(f"{V:>6.2f} {abs(_s):>14.4f} {TSIRELSON_BOUND*V:>10.4f} {_q:>15.4f} {(1-V)/2:>10.4f}")

    print("\nSingle-run feature vectors, one per attack mode:\n")
    _demo_feats = {}
    for mode in ["none", "intercept_resend", "ancilla", "loss_manipulation"]:
        f = extract_e91_features(n_pulses=200_000, eve_mode=mode, n_windows=20,
                                  rng=np.random.default_rng(42), eve_intensity=0.5, V=0.95)
        _demo_feats[mode] = f
        print(f"--- eve_mode = {mode} ---")
        for k, v in f.items():
            print(f"  {k:26s}: {v:.4f}" if isinstance(v, float) else f"  {k:26s}: {v}")
        print()

    print(f"chsh_S -- none={_demo_feats['none']['chsh_S']:.4f}  "
          f"intercept_resend={_demo_feats['intercept_resend']['chsh_S']:.4f}  "
          f"ancilla={_demo_feats['ancilla']['chsh_S']:.4f}  "
          f"loss_manipulation={_demo_feats['loss_manipulation']['chsh_S']:.4f}  "
          f"(Tsirelson bound = {TSIRELSON_BOUND:.4f}, classical bound = 2)")
    print(f"s_qber_residual -- none={_demo_feats['none']['s_qber_residual']:.4f}  "
          f"intercept_resend={_demo_feats['intercept_resend']['s_qber_residual']:.4f}  "
          f"ancilla={_demo_feats['ancilla']['s_qber_residual']:.4f}  "
          "(expect ancilla >> the other two -- it is basis-anisotropic, they are not)")
    assert abs(_demo_feats['ancilla']['s_qber_residual']) > abs(_demo_feats['none']['s_qber_residual']), "ancilla attack should show a larger s_qber_residual than honest noise"
    print("PASS -- simulator behaves correctly, safe to build features on top of it.")

In [ ]:
# ─── Full feature-extraction functions ───────────────────────────────────────

# P12: BB84_FEATURE_NAMES replaces the old 10-feature list.
#   REMOVED: multi_rate -- it was a perfect label indicator (photon
#     statistics were only ever sampled for the PNS class) and is in any
#     case not an observable available to Bob (he cannot count photons).
#   RENAMED: holevo_ie -> h_qber -- it is the binary Shannon entropy h(Q),
#     not Holevo information (r=0.99 with qber_total -- near-redundant).
#   REPLACED: qber_variance -> qber_dispersion (ratio to the binomial
#     floor; not confounded with the mean, unlike raw variance).
#   ADDED: decoy-state observables (y1_lower, e1_upper, r_secure,
#     gain_ratio_nu_mu) -- the physically correct PNS detectors.
BB84_FEATURE_NAMES = [
    'qber_total', 'qber_z', 'qber_x',
    'gain_mu', 'sifted_rate',
    'qber_dispersion', 'jump_energy_norm', 'spectral_entropy', 'autocorr_lag1',
    'y1_lower', 'e1_upper', 'r_secure', 'gain_ratio_nu_mu',
    'h_qber',
]


def collect_bb84_features(N, distance_km, eve_mode='none', eve_intensity=0.0,
                           profile='iid', pns_strategy=None, n_windows=64,
                           rng=None, **chan):
    '''Run N BB84 pulses through the physical channel (P1/P2) and compress
    into the BB84_FEATURE_NAMES feature vector, via the vectorised
    decoy-state simulator (P5).'''
    rng = rng or np.random.default_rng()
    run = simulate_bb84_decoy(N, distance_km, eve_mode, eve_intensity,
                               profile=profile, pns_strategy=pns_strategy,
                               rng=rng, **chan)

    bit_A, bit_B = run['bit_A'], run['bit_B']
    bas_A, sift = run['bas_A'], run['sift']
    is_sig = (run['k'] == 0)          # signal-intensity pulses only
    s = sift & is_sig
    sz = s & (bas_A == 0); sx = s & (bas_A == 1)

    qber_total = float((bit_A[s] != bit_B[s]).mean()) if s.any() else 0.0
    qber_z = float((bit_A[sz] != bit_B[sz]).mean()) if sz.any() else 0.0
    qber_x = float((bit_A[sx] != bit_B[sx]).mean()) if sx.any() else 0.0
    gain_mu = run['Q'][run['intensities'][0]]
    sifted_rate = float(s.sum() / max(is_sig.sum(), 1))

    # P11: window loop keeps (error_count, sifted_count) per window, not a
    # bare ratio -- the dispersion index needs the counts to compute the
    # binomial floor. Parameterised by NUMBER OF WINDOWS, not pulses per
    # window, so a fixed window count gives comparable estimator variance
    # regardless of distance/loss.
    edges = np.linspace(0, N, n_windows + 1, dtype=int)
    w_err, w_cnt = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = s[lo:hi]
        w_cnt.append(int(m.sum()))
        w_err.append(int((bit_A[lo:hi][m] != bit_B[lo:hi][m]).sum()))
    w_err, w_cnt = np.array(w_err), np.array(w_cnt)
    with np.errstate(invalid='ignore', divide='ignore'):
        wq = np.where(w_cnt > 0, w_err / np.maximum(w_cnt, 1), np.nan)

    # Decoy-state block (P5's estimators)
    mu_s, mu_d, mu_v = run['intensities']
    Y0_obs = run['Q'][mu_v]
    y1, q1, e1 = decoy_estimate(run['Q'][mu_s], run['E'][mu_s],
                                 run['Q'][mu_d], run['E'][mu_d],
                                 Y0_obs, mu=mu_s, nu=mu_d)
    r_sec = secure_key_rate(run['Q'][mu_s], run['E'][mu_s], q1, e1)
    ratio = float((run['Q'][mu_d] * np.exp(mu_d)) /
                  max(run['Q'][mu_s] * np.exp(mu_s), 1e-18))

    return {
        'qber_total': qber_total,
        'qber_z': qber_z,
        'qber_x': qber_x,
        'gain_mu': float(gain_mu),
        'sifted_rate': sifted_rate,
        'qber_dispersion': _dispersion_index(w_err, w_cnt),
        'jump_energy_norm': _jump_energy_norm(w_err, w_cnt),
        'spectral_entropy': _spectral_entropy(wq),
        'autocorr_lag1': _autocorr_lag1(wq[~np.isnan(wq)]),
        'y1_lower': float(y1),
        'e1_upper': float(e1),
        'r_secure': float(r_sec),
        'gain_ratio_nu_mu': ratio,
        'h_qber': float(_binary_entropy(qber_total)),
    }


def collect_bkm07_features(N, distance_km, eve_mode, eve_fwd, eve_ret,
                            n_windows=64, rng=None, **chan):
    '''Run N BKM07 round trips and compress into a feature vector.

    P10 changes: Alice measures SIFT returns in her OWN preparation basis
    (not always Z); X-SIFT rounds are protocol-discarded and kept only as
    an auxiliary `ret_monitor` (renamed from `qber_xsr`, which the previous
    version reported as if it were a protocol observable -- it was not:
    Bob re-prepares in Z, Alice measured in Z, so it measured only
    return-leg channel noise); the channel is applied TWICE (round trip:
    t^2, not t) via simulate_bkm07_pulse's per-leg loss; `xctrl_variance`
    replaced by a dispersion index (P6/P11).
    '''
    rng = rng or np.random.default_rng()
    pulses = [simulate_bkm07_pulse(distance_km, eve_mode, eve_fwd, eve_ret, rng=rng, **chan)
              for _ in range(N)]
    pulses = [p for p in pulses if not p.get('lost', False)]

    z_sft_t, z_sft_e, z_sft_ret_e, z_sft_key_e = 0, 0, 0, 0
    x_sft_t, x_sft_ret_e = 0, 0
    z_ctrl_t, z_ctrl_e = 0, 0
    x_ctrl_t, x_ctrl_e = 0, 0

    for p in pulses:
        rt = p['round_type']
        if rt == 'SIFT_KEY':
            z_sft_t += 1
            z_sft_e += int(p['bit_A'] != p['bit_B'])
            z_sft_ret_e += int(p['bit_B'] != p['bit_A_final'])
            z_sft_key_e += int(p['bit_A'] != p['bit_A_final'])
        elif rt == 'SIFT_MONITOR':
            x_sft_t += 1
            x_sft_ret_e += int(p['bit_B'] != p['bit_A_final'])
        elif rt == 'CTRL_Z':
            z_ctrl_t += 1
            z_ctrl_e += int(p['bit_A'] != p['bit_A_final'])
        elif rt == 'CTRL_X':
            x_ctrl_t += 1
            x_ctrl_e += int(p['bit_A'] != p['bit_A_final'])

    qber_zs = z_sft_e / z_sft_t if z_sft_t > 0 else 0.0
    # True end-to-end round-trip key error (bit_A vs bit_A_final): the
    # quantity a real BKM07 deployment reports as key QBER. qber_zs above
    # is NOT this -- it's a 2-flip composite (Bob's measurement noise +
    # his re-preparation flip) because `bit_B` is reassigned after the
    # prep-flip line in simulate_bkm07_pulse. This was missing before.
    qber_key = z_sft_key_e / z_sft_t if z_sft_t > 0 else 0.0
    qber_zsr = z_sft_ret_e / z_sft_t if z_sft_t > 0 else 0.0
    ret_monitor = x_sft_ret_e / x_sft_t if x_sft_t > 0 else 0.0   # was qber_xsr
    qber_zc = z_ctrl_e / z_ctrl_t if z_ctrl_t > 0 else 0.0
    qber_xc = x_ctrl_e / x_ctrl_t if x_ctrl_t > 0 else 0.0

    asymmetry = abs(qber_zsr - qber_zs)
    ctrl_sift_ratio = float((qber_zc + qber_xc) / 2.0) / max(qber_zs, 1e-9)

    N_eff = max(len(pulses), 1)
    edges = np.linspace(0, N_eff, n_windows + 1, dtype=int)
    w_err, w_cnt = [], []
    idx = 0
    for lo, hi in zip(edges[:-1], edges[1:]):
        chunk = pulses[lo:hi]
        wt, we = 0, 0
        for p in chunk:
            if p['round_type'] == 'CTRL_X':
                wt += 1
                we += int(p['bit_A'] != p['bit_A_final'])
        w_cnt.append(wt); w_err.append(we)
    w_err, w_cnt = np.array(w_err), np.array(w_cnt)
    with np.errstate(invalid='ignore', divide='ignore'):
        wq = np.where(w_cnt > 0, w_err / np.maximum(w_cnt, 1), np.nan)

    qber_ctrl_avg = (qber_zc + qber_xc) / 2.0

    return {
        'qber_key': qber_key,
        'qber_zs': qber_zs,
        'qber_zsr': qber_zsr,
        'ret_monitor': ret_monitor,     # was 'qber_xsr' (P10 rename)
        'qber_zc': qber_zc,
        'qber_xc': qber_xc,
        'asymmetry': asymmetry,
        'ctrl_sift_ratio': ctrl_sift_ratio,
        'xctrl_dispersion': _dispersion_index(w_err, w_cnt),
        'jump_energy': _jump_energy(wq[~np.isnan(wq)]),
        'spectral_entropy': _spectral_entropy(wq),
        'autocorr_lag1': _autocorr_lag1(wq[~np.isnan(wq)]),
        'h_ctrl': float(_binary_entropy(qber_ctrl_avg)),   # was 'holevo_ie'
        'sifted_rate': z_sft_t / N_eff,
    }


BKM_FEATURE_NAMES = ['qber_key', 'qber_zs', 'qber_zsr', 'ret_monitor', 'qber_zc', 'qber_xc',
                      'asymmetry', 'ctrl_sift_ratio', 'xctrl_dispersion',
                      'jump_energy', 'spectral_entropy', 'autocorr_lag1',
                      'h_ctrl', 'sifted_rate']

print("Feature-extraction functions defined (P11/P12 BB84 decoy+dispersion; P10 BKM07 round-fix).")

### Matched-QBER calibration layer

`distance_km` (BB84/BKM07) and `V` (E91) are not comparable knobs: BB84's
honest QBER rises nonlinearly with distance (the GYS closed form),
E91's is linear in `(1-V)`, and — as derived below — BKM07's honest QBER
turns out to be **independent of `distance_km` entirely** in this model
(distance only sets round-trip survival probability via `eta_1way**2`, not
the per-round error rate). Sweeping the raw knob and comparing AUC across
protocols would mostly report how each knob happens to map to QBER, not
anything about the protocols themselves. Instead, calibrate each
protocol's true noise-generating parameter to hit a shared **target honest
QBER**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Matched-QBER calibration: invert each protocol's honest-noise model so
# BB84 (distance_km), E91 (V) and BKM07 (e_detector) can all be driven to
# the SAME target honest QBER for a fair cross-protocol comparison.
# ═══════════════════════════════════════════════════════════════════════════

def calibrate_bb84(target_qber, mu=MU_SIGNAL, alpha_db_km=None, eta_bob=None,
                    Y0=None, e_detector=None, e_0=None):
    """Exact algebraic inverse of channel_model()'s E_mu (Ma et al. 2005,
    Eq. 11): solve E_mu(distance_km) = target_qber for distance_km.
    Only defined for e_detector < target_qber < e_0 -- outside that range
    no fibre distance can produce it (the channel saturates at e_detector
    near 0km and at e_0 as distance -> infinity, exactly the floor/ceiling
    the audit table shows)."""
    alpha_db_km = GYS['alpha_db_km'] if alpha_db_km is None else alpha_db_km
    eta_bob = GYS['eta_bob'] if eta_bob is None else eta_bob
    Y0 = GYS['Y0'] if Y0 is None else Y0
    e_detector = GYS['e_detector'] if e_detector is None else e_detector
    e_0 = GYS['e_0'] if e_0 is None else e_0

    if not (e_detector < target_qber < e_0):
        raise ValueError(
            f"target_qber={target_qber} unreachable for BB84/GYS: honest "
            f"QBER only spans (e_detector={e_detector:.4f}, e_0={e_0:.4f}) "
            f"as distance_km spans (0, inf).")

    p_signal = Y0 * (e_0 - target_qber) / (target_qber - e_detector)
    eta = -np.log(1.0 - p_signal) / mu
    t_AB = eta / eta_bob
    distance_km = -10.0 * np.log10(t_AB) / alpha_db_km
    return float(distance_km)


def calibrate_e91(target_qber):
    """Exact: Werner-state honest key QBER = (1-V)/2 (see werner_state())."""
    if not (0.0 < target_qber < 0.5):
        raise ValueError(f"target_qber={target_qber} unreachable for E91: "
                          f"honest QBER spans (0, 0.5) as V spans (1, 0).")
    return float(1.0 - 2.0 * target_qber)


def calibrate_bkm07(target_qber, e_detector_bounds=(1e-6, 0.499)):
    """Closed-form inverse of BKM07's honest end-to-end key QBER
    (`qber_key`: bit_A vs bit_A_final over SIFT_KEY rounds -- NOT `qber_zs`,
    which is a 2-flip composite; see collect_bkm07_features()).

    A SIFT_KEY round trip passes through THREE independent noisy
    operations (Bob's SIFT measurement, Bob's re-preparation flip, Alice's
    final measurement -- see simulate_bkm07_pulse), each an independent
    binary symmetric channel with crossover probability e_detector.
    Composing n independent BSCs of crossover e gives combined crossover
    0.5*(1-(1-2e)**n); for n=3 that inverts to the closed form below.

    This makes NO reference to distance_km: in the current model, distance
    only sets round-trip SURVIVAL probability (eta_1way**2, via the two
    loss checks in simulate_bkm07_pulse), not the per-round error rate --
    e_detector is the only knob that moves honest QBER. So unlike BB84
    (where distance IS the QBER knob), calibrating BKM07 means solving for
    e_detector; distance_km is left free for whoever calls this, to
    separately match round-trip yield/sifted_rate if a matched-throughput
    comparison is also wanted.
    """
    if not (0.0 < target_qber < 0.5):
        raise ValueError(f"target_qber={target_qber} unreachable for BKM07: "
                          f"SIFT_KEY QBER spans (0, 0.5) as e_detector spans (0, 0.5).")
    e_detector = (1.0 - (1.0 - 2.0 * target_qber) ** (1.0 / 3.0)) / 2.0
    lo, hi = e_detector_bounds
    return float(np.clip(e_detector, lo, hi))


# ── Validation: each calibration should reproduce its own target QBER ─────
print("Closed-form self-check (each protocol's own honest-noise formula):")
print(f"{'target_qber':>12} | {'BB84 dist_km':>12} {'-> qber':>9} | "
      f"{'E91 V':>7} {'-> qber':>9} | {'BKM07 e_det':>12} {'-> qber':>9}")
for q in [0.02, 0.033, 0.05, 0.08, 0.10, 0.15]:
    try:
        d = calibrate_bb84(q)
        bb84_check = channel_model(d)['qber']
        d_str, bb84_str = f"{d:>12.2f}", f"{bb84_check:>9.4f}"
    except ValueError:
        d_str, bb84_str = f"{'n/a':>12}", f"{'--':>9}"
    V = calibrate_e91(q)
    e91_check = (1.0 - V) / 2.0
    ed = calibrate_bkm07(q)
    bkm_check = 0.5 * (1.0 - (1.0 - 2.0 * ed) ** 3)
    print(f"{q:>12.3f} | {d_str} {bb84_str} | "
          f"{V:>7.4f} {e91_check:>9.4f} | {ed:>12.4f} {bkm_check:>9.4f}")

# ── Empirical cross-check: does the ACTUAL stochastic BKM07 simulator
# (not just the hand-derived closed form) reproduce the target QBER when
# calibrated? This is the one derivation above that isn't a one-line
# algebraic identity, so it gets its own Monte-Carlo confirmation.
#
# distance_km=0.0 is deliberate, not a shortcut: round-trip survival is
# eta_1way**2, and eta_1way = t_AB * eta_bob is already <= eta_bob = 0.045
# even at zero fibre length (Bob's finite detector efficiency), so ANY
# distance_km leaves few SIFT_KEY survivors per pulse fired. Since the
# claim under test is specifically that distance_km does NOT affect honest
# QBER, testing at its most favourable (lowest-loss) value maximises the
# effective sample size without changing what's being validated.
print("\nEmpirical check -- collect_bkm07_features() at the calibrated e_detector:")
# Round-trip survival at distance_km=0.0 is eta_bob**2 ~ 0.002 (BOTH loss
# checks in simulate_bkm07_pulse must pass), and SIFT_KEY rounds are ~1/4
# of survivors (P(SIFT)=0.5 x P(basis_A==0)=0.5) -- so most fired pulses
# never reach a SIFT_KEY round. n_eff_theory below is the expected survivor
# count from that model, used only to size the reported CI -- NOT
# collect_bkm07_features()'s own 'sifted_rate' field, which is a fraction
# of SURVIVORS, not of N_CHECK (multiplying that by N_CHECK overstates
# n_eff by ~1/eta**2 and was the source of an earlier spurious "mismatch").
N_CHECK = 3_000_000
for q in [0.05, 0.10]:
    ed = calibrate_bkm07(q)
    eta = channel_model(0.0, e_detector=ed)['eta']
    n_eff_theory = N_CHECK * eta ** 2 * 0.25
    f = collect_bkm07_features(N_CHECK, distance_km=0.0, eve_mode='none',
                                eve_fwd=0.0, eve_ret=0.0, e_detector=ed,
                                rng=np.random.default_rng(0))
    p_hat = f['qber_key']
    n_int = max(round(n_eff_theory), 1)
    e_int = round(p_hat * n_int)
    _, ci_lo, ci_hi = qber_ci(e_int, n_int)   # exact Clopper-Pearson binomial CI
    print(f"  target={q:.3f}  e_detector={ed:.4f}  n_eff(theory)~={n_eff_theory:7.0f}  "
          f"measured qber_key={p_hat:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")

---
## Section 4 — Generating the Labelled Dataset

We generate `samples_per_class` independent simulated runs for each of the six scenario classes:

| Protocol | Label | Scenario |
|---|---|---|
| BB84 | 0 | Secure (no Eve) |
| BB84 | 1 | Intercept-resend attack |
| BB84 | 2 | Photon-number-splitting (PNS) attack |
| BKM07 | 0 | Secure (no Eve) |
| BKM07 | 1 | Symmetric attack (equal forward/return intensity) |
| BKM07 | 2 | Asymmetric attack (weak forward, strong return, or vice versa) |

For each run index, a fiber distance and a set of nuisance channel parameters (`e_detector`, `Y0`) are drawn **once** and reused across that index's secure/attacked samples (common random numbers, P13) — this is what lets the leakage audit's nuisance-parameter check (Section "Leakage Audit", L-4) pass honestly, since the channel realisation can no longer leak the label. BB84's distance spans the full `CHANNEL_DISTANCE_RANGE_KM = (0, 100)` km; BKM07's is narrower, `DISTANCE_RANGE_BKM = (0, 15)` km, because round-trip loss (`eta_1way**2`) makes longer links yield too few surviving round trips per run to be useful.

The datasets are also written to `data/bb84_dataset.csv` and `data/bkm07_dataset.csv`.

> **Note on scale/runtime:** the default `samples_per_class=60`, `N=2_000_000` (BB84), `N_bkm=20_000` (BKM07) is already a *reduction* from what the audit recommends for a publication-grade run (larger `samples_per_class` narrows the wide confidence intervals seen in Section 9's leave-one-cell-out results) — it's sized to keep the whole notebook runnable end-to-end in a few minutes. Expect this cell alone to take 1-3 minutes.

In [ ]:
# BKM07 is a round-trip protocol (loss applies TWICE), so realistic
# semi-quantum systems are inherently short-range; keep its distance
# draw narrower than BB84's so N_bkm pulses still yield enough surviving
# round trips per window (audit Sec. G.3.3).
DISTANCE_RANGE_BKM = (0.0, 15.0)


def generate_datasets(samples_per_class=60, N=2_000_000, N_bkm=20_000, n_windows=64):
    '''Simulate QKD runs for all attack classes and return feature arrays.

    P16 fix (was the leakage audit's L-4 failure): nuisance parameters
    (distance_km, e_detector, Y0) are drawn ONCE per run index, BEFORE the
    class loop, and the SAME values are used for every class at that index.
    Previously PNS runs drew mu~U(0.08,0.25) while other classes used the
    default mu=0.1 -- a classifier could tell the classes apart from the
    nuisance-parameter distribution alone, with no attack detection
    involved. This is also "common random numbers" (P13): pairing honest
    and attacked runs on the same channel realisation lowers the variance
    of any comparison between them.

    N defaults to 2,000,000 pulses/run: with the real channel model (P1)
    and loss (P2), a realistic link only sifts on the order of a few
    hundred to a few thousand bits per run at N~2000-10000 (see audit Sec.
    H.2), which is nowhere near enough for the window-based temporal
    features to mean anything.
    '''
    bb84_rows, bkm_rows = [], []

    print("  [BB84] Generating samples for all classes (shared nuisance draws per run) ...")
    for i in range(samples_per_class):
        nuis_rng = SEEDS.rng('bb84_nuisance', i)
        distance_km = float(nuis_rng.uniform(*CHANNEL_DISTANCE_RANGE_KM))
        e_det = float(np.clip(nuis_rng.normal(0.033, 0.005), 0.01, 0.05))
        Y0 = float(10 ** nuis_rng.uniform(-6, -5))
        strat = make_pns_strategy(distance_km, e_detector=e_det, Y0=Y0)

        rng = SEEDS.rng('bb84_none', i)
        f = collect_bb84_features(N, distance_km, 'none', 0.0, n_windows=n_windows,
                                   rng=rng, e_detector=e_det, Y0=Y0)
        bb84_rows.append([f[k] for k in BB84_FEATURE_NAMES] + [distance_km, 0.0, 0])

        rng = SEEDS.rng('bb84_ir', i)
        di = float(rng.uniform(0.05, 1.0))
        profile = str(rng.choice(['iid', 'bursty']))
        f = collect_bb84_features(N, distance_km, 'intercept_resend', di, profile=profile,
                                   n_windows=n_windows, rng=rng, e_detector=e_det, Y0=Y0)
        bb84_rows.append([f[k] for k in BB84_FEATURE_NAMES] + [distance_km, di, 1])

        rng = SEEDS.rng('bb84_pns', i)
        pi = float(rng.uniform(0.3, 1.0))
        f = collect_bb84_features(N, distance_km, 'pns', pi, pns_strategy=strat,
                                   n_windows=n_windows, rng=rng, e_detector=e_det, Y0=Y0)
        bb84_rows.append([f[k] for k in BB84_FEATURE_NAMES] + [distance_km, pi, 2])

        if (i + 1) % max(1, samples_per_class // 10) == 0:
            print(f"    {i + 1}/{samples_per_class} runs", flush=True)

    print("  [BKM07] Generating samples for all classes (shared nuisance draws per run) ...")
    for i in range(samples_per_class):
        nuis_rng = SEEDS.rng('bkm_nuisance', i)
        distance_km = float(nuis_rng.uniform(*DISTANCE_RANGE_BKM))
        e_det = float(np.clip(nuis_rng.normal(0.033, 0.005), 0.01, 0.05))

        rng = SEEDS.rng('bkm_none', i)
        f = collect_bkm07_features(N_bkm, distance_km, 'none', 0.0, 0.0,
                                    n_windows=n_windows, rng=rng, e_detector=e_det)
        bkm_rows.append([f[k] for k in BKM_FEATURE_NAMES] + [distance_km, 0.0, 0.0, 0])

        rng = SEEDS.rng('bkm_sym', i)
        di = float(rng.uniform(0.05, 0.5))
        f = collect_bkm07_features(N_bkm, distance_km, 'symmetric', di, di,
                                    n_windows=n_windows, rng=rng, e_detector=e_det)
        bkm_rows.append([f[k] for k in BKM_FEATURE_NAMES] + [distance_km, di, di, 1])

        rng = SEEDS.rng('bkm_asym', i)
        di_fwd = float(rng.uniform(0.02, 0.2))
        di_ret = float(rng.uniform(0.2, 0.6))
        f = collect_bkm07_features(N_bkm, distance_km, 'asymmetric', di_fwd, di_ret,
                                    n_windows=n_windows, rng=rng, e_detector=e_det)
        bkm_rows.append([f[k] for k in BKM_FEATURE_NAMES] + [distance_km, di_fwd, di_ret, 2])

        if (i + 1) % max(1, samples_per_class // 10) == 0:
            print(f"    {i + 1}/{samples_per_class} runs", flush=True)

    bb84_header = BB84_FEATURE_NAMES + ['distance_km', 'eve_intensity', 'label']
    bkm_header = BKM_FEATURE_NAMES + ['distance_km', 'eve_fwd', 'eve_ret', 'label']

    with open('data/bb84_dataset.csv', 'w', newline='') as fh:
        csv.writer(fh).writerows([bb84_header] + bb84_rows)
    with open('data/bkm07_dataset.csv', 'w', newline='') as fh:
        csv.writer(fh).writerows([bkm_header] + bkm_rows)

    return (np.array(bb84_rows, dtype=float), np.array(bkm_rows, dtype=float),
            bb84_header, bkm_header)


print("generate_datasets() defined -- running now ...")
bb84_arr, bkm_arr, bb84_hdr, bkm_hdr = generate_datasets(samples_per_class=60, N=2_000_000, N_bkm=20_000)

bb84_attack_labels = np.array(['none', 'intercept_resend', 'pns'])[bb84_arr[:, -1].astype(int)]

print(f"\nBB84 dataset shape : {bb84_arr.shape}   "
      f"(label counts: {dict(zip(*np.unique(bb84_arr[:,-1].astype(int), return_counts=True)))})")
print(f"BKM07 dataset shape: {bkm_arr.shape}   "
      f"(label counts: {dict(zip(*np.unique(bkm_arr[:,-1].astype(int), return_counts=True)))})")
print("CSVs saved to data/")

In [ ]:
# ── P16: leakage audit -- run BEFORE training any real model ───────────────
# Kapoor & Narayanan, Patterns 4, 100804 (2023). Eight tests; L-8
# (zero-strength control) is the single most valuable one: it keeps the
# class LABELS but sets every attack strength to zero, so AUC must fall to
# ~0.50. If it does not, the label is leaking through something other than
# the simulated attack.
def leakage_audit(X, y, feature_names, attack_labels, nuisance, N_per_run):
    print("=" * 74); print("LEAKAGE AUDIT"); print("=" * 74)

    print("\nL-1 Single-feature AUC (>0.99 is a red flag)")
    for i, nme in enumerate(feature_names):
        col = X[:, i]; ok = np.isfinite(col)
        if ok.sum() < 10 or len(np.unique(y[ok])) < 2:
            continue
        a = roc_auc_score(y[ok], col[ok]); a = max(a, 1 - a)
        flag = " <-- LEAK?" if a > 0.99 else ""
        print(f"  {nme:24s} AUC={a:.4f}{flag}")

    print("\nL-2/L-3 Exact separation and per-class constants")
    for i, nme in enumerate(feature_names):
        for cls in np.unique(attack_labels):
            v = X[attack_labels == cls, i]; v = v[np.isfinite(v)]
            if len(v) > 5 and np.ptp(v) == 0:
                print(f"  {nme:24s} is CONSTANT ({v[0]:.6g}) for class '{cls}' <-- LEAK")
        a = X[attack_labels == 'none', i]; b = X[attack_labels != 'none', i]
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        if len(a) and len(b) and (a.max() < b.min() or b.max() < a.min()):
            print(f"  {nme:24s} classes are DISJOINT <-- LEAK")

    print("\nL-4 Nuisance-parameter distributions must match across classes (KS test)")
    for pname, pvals in nuisance.items():
        base = pvals[attack_labels == 'none']
        for cls in np.unique(attack_labels):
            if cls == 'none':
                continue
            other = pvals[attack_labels == cls]
            if len(base) < 3 or len(other) < 3:
                continue
            ks, p = stats.ks_2samp(base, other)
            flag = " <-- CLASSES DIFFER" if p < 0.01 else ""
            print(f"  {pname:16s} none vs {cls:20s} KS={ks:.3f} p={p:.3g}{flag}")

    print("\nL-5 Sample length identical across classes")
    for cls in np.unique(attack_labels):
        v = np.unique(N_per_run[attack_labels == cls])
        print(f"  {cls:24s} N values = {v}")

    print("\nL-6 Feature availability (NaN rate) by class")
    for i, nme in enumerate(feature_names):
        rates = {c: float(np.mean(~np.isfinite(X[attack_labels == c, i])))
                 for c in np.unique(attack_labels)}
        if max(rates.values()) - min(rates.values()) > 0.05:
            print(f"  {nme:24s} NaN rate varies by class: {rates} <-- LEAK")

    print(); print("L-7 Shuffled-label control (AUC must be ~0.50)")
    # Uses a small, regularised Logistic Regression (median-impute + scale)
    # rather than a tree ensemble: on the small sample sizes used here, a
    # flexible booster can overfit pure noise and give a spuriously elevated
    # AUC even with correctly shuffled labels -- a small-N artefact of the
    # audit probe itself, not evidence of a leak. Averaged over 5 shuffles.
    def _audit_probe():
        return Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler()),
                          ('clf', LogisticRegression(C=0.1, max_iter=2000))])
    _aucs = []
    for _seed in range(5):
        rs = np.random.RandomState(_seed); yp = rs.permutation(y)
        perm = rs.permutation(len(y)); sp = int(0.8 * len(y))
        m = _audit_probe()
        m.fit(X[perm[:sp]], yp[perm[:sp]])
        _aucs.append(roc_auc_score(yp[perm[sp:]], m.predict_proba(X[perm[sp:]])[:, 1]))
    a = float(np.mean(_aucs))
    print(f"  shuffled-label AUC = {a:.4f} (range {min(_aucs):.3f}-{max(_aucs):.3f} over 5 shuffles) "
          f"{'OK' if abs(a - 0.5) < 0.08 else '<-- PIPELINE BUG'}")
    print("=" * 74)
    return a


def zero_strength_control_bb84(n_per_class=80, N=1_000_000, n_windows=32, distance_km=25.0):
    """L-8: keep the class LABELS but set every attack strength to zero.
    AUC must fall to ~0.50. This is the single most valuable control --
    if it does not, the label leaks through something other than the
    attack itself."""
    rows, labs = [], []
    e_det = 0.033
    strat = make_pns_strategy(distance_km, e_detector=e_det)
    for cls in ['none', 'intercept_resend', 'pns']:
        for i in range(n_per_class):
            rng = SEEDS.rng('l8_control_' + cls, i)
            eve_mode = cls
            f = collect_bb84_features(N, distance_km, eve_mode=eve_mode,
                                       eve_intensity=0.0,   # <-- ZERO, but label kept
                                       pns_strategy=strat, n_windows=n_windows,
                                       rng=rng, e_detector=e_det)
            rows.append([f[k] for k in BB84_FEATURE_NAMES])
            labs.append(cls)
    Xc = np.array(rows, dtype=float); yc = (np.array(labs) != 'none').astype(int)
    # Same small-N-robust probe as L-7 (see note there), averaged over
    # several splits.
    def _audit_probe():
        return Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler()),
                          ('clf', LogisticRegression(C=0.1, max_iter=2000))])
    aucs = []
    for seed in range(5):
        perm = np.random.RandomState(seed).permutation(len(Xc)); sp = int(0.8 * len(perm))
        m = _audit_probe()
        m.fit(Xc[perm[:sp]], yc[perm[:sp]])
        aucs.append(roc_auc_score(yc[perm[sp:]], m.predict_proba(Xc[perm[sp:]])[:, 1]))
    a = float(np.mean(aucs))
    print(f"L-8 zero-strength control AUC = {a:.4f} (range {min(aucs):.3f}-{max(aucs):.3f} over 5 splits) "
          f"{'OK' if abs(a - 0.5) < 0.1 else '<-- LABEL LEAKS'}")
    return a


N_FEAT_84_PROBE = len(BB84_FEATURE_NAMES)
bb84_nuisance = {'distance_km': bb84_arr[:, N_FEAT_84_PROBE]}
bb84_N_per_run = np.full(len(bb84_arr), 2_000_000)

leakage_audit(bb84_arr[:, :N_FEAT_84_PROBE], (bb84_arr[:, -1] > 0).astype(int),
              BB84_FEATURE_NAMES, bb84_attack_labels, bb84_nuisance, bb84_N_per_run)

zero_strength_control_bb84()

### Train / Test Split

We binarise the labels (0 = secure, 1 = any attack present) and split 80 / 20.

A fixed `RandomState(7)` is used so the split is reproducible across notebook runs.


In [ ]:
N_FEAT_84 = len(BB84_FEATURE_NAMES)
N_FEAT_BK = len(BKM_FEATURE_NAMES)

bb84_X = bb84_arr[:, :N_FEAT_84].astype(float)
bkm_X = bkm_arr[:, :N_FEAT_BK].astype(float)

# Defensive median imputation: a handful of BKM07 runs at the high end
# of its distance range can still leave a window with zero CTRL_X
# rounds (giving NaN dispersion/temporal features); SVM/RF cannot accept
# NaN. Impute per-column median (a pre-split defensive cleanup on a
# structurally-missing value, not a leakage-sensitive statistic).
for _X in (bb84_X, bkm_X):
    _col_median = np.nanmedian(_X, axis=0)
    _col_median = np.where(np.isfinite(_col_median), _col_median, 0.0)
    _nan_mask = ~np.isfinite(_X)
    if _nan_mask.any():
        _X[_nan_mask] = np.take(_col_median, np.where(_nan_mask)[1])

bb84_y = (bb84_arr[:, -1].astype(int) > 0).astype(int)    # 0=secure, 1=Eve
bb84_noise = bb84_arr[:, N_FEAT_84].astype(float)          # distance_km
bb84_eve_intensity_col = bb84_arr[:, N_FEAT_84 + 1].astype(float)

bkm_y = (bkm_arr[:, -1].astype(int) > 0).astype(int)

rng = np.random.RandomState(7)

bb84_idx = rng.permutation(len(bb84_X))
sp84 = int(0.8 * len(bb84_X))
X84tr, y84tr = bb84_X[bb84_idx[:sp84]], bb84_y[bb84_idx[:sp84]]
X84te, y84te = bb84_X[bb84_idx[sp84:]], bb84_y[bb84_idx[sp84:]]

bkm_idx = rng.permutation(len(bkm_X))
spbk = int(0.8 * len(bkm_X))
Xbktr, ybktr = bkm_X[bkm_idx[:spbk]], bkm_y[bkm_idx[:spbk]]
Xbkte, ybkte = bkm_X[bkm_idx[spbk:]], bkm_y[bkm_idx[spbk:]]

print(f"BB84  — train: {X84tr.shape}, test: {X84te.shape}")
print(f"BKM07 — train: {Xbktr.shape}, test: {Xbkte.shape}")
print(f"BB84  class balance in train: {dict(zip(*np.unique(y84tr, return_counts=True)))}")
print(f"BKM07 class balance in train: {dict(zip(*np.unique(ybktr, return_counts=True)))}")

---
## Section 5 — Machine Learning Models

We use six classifiers. All supervised models are wrapped in `Pipeline(StandardScaler → model)` so that feature scaling is fitted only on training data (preventing leakage into cross-validation folds).

| Model | Key idea | Tuned? |
|---|---|---|
| **KNN** (k=7) | Classify by majority vote among 7 nearest training neighbours | No |
| **Logistic Regression** | Single (approximately linear) decision boundary | No |
| **Random Forest** | Ensemble of many decision trees, majority vote | **Yes** (CV) |
| **SVM-RBF** | Maximum-margin hyperplane with RBF kernel (handles non-linear boundaries) | **Yes** (CV) |
| **XGBoost / HistGB** | Boosted trees: each tree corrects the previous ones' mistakes | **Yes** (CV) |
| **Isolation Forest** | *Unsupervised* anomaly detector — trained only on secure data, flags anything that looks unusual | No |

### Why tune only three?

KNN and LogReg are simple enough that default settings are close to optimal on this dataset. The three tree-based / kernel models benefit significantly from tuning and have more budget-sensitive hyperparameters.

### Hyperparameter tuning strategy

We use `GridSearchCV` with 5-fold *stratified* cross-validation, optimising **ROC-AUC** rather than accuracy. AUC is the right metric here because:
- It is threshold-independent (we may want to adjust the detection threshold in practice).
- It is insensitive to class imbalance (not an issue here, but good practice).


In [ ]:
# ─── Model factory functions ─────────────────────────────────────────────────

def make_knn(k=7):
    return Pipeline([('scaler', StandardScaler()),
                     ('clf', KNeighborsClassifier(n_neighbors=k))])

def make_logreg():
    return Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(max_iter=2000))])

def make_boosted(seed=0, **params):
    '''XGBoost if available, else HistGradientBoostingClassifier.
    Both use the same gradient-boosting algorithm and produce very
    similar results. `subsample` and `colsample_bytree` add stochastic
    regularisation (XGBoost only).'''
    if HAS_XGB:
        defaults = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                        subsample=0.9, colsample_bytree=0.9,
                        eval_metric='logloss', random_state=seed)
        defaults.update(params)
        return XGBClassifier(**defaults)
    else:
        defaults = dict(max_iter=300, max_depth=5, learning_rate=0.05,
                        random_state=seed)
        defaults.update(params)
        return HistGradientBoostingClassifier(**defaults)

def make_isolation_forest(max_samples=256, n_estimators=200, seed=1):
    '''Isolation Forest: anomaly detection without labels.
    Trains only on 'normal' (secure) data. At inference time it gives
    each sample an anomaly score based on how quickly it gets isolated
    by random splits — unusual samples get isolated quickly (high score).
    max_samples=256 is the subsample size per tree (controls variance).
    '''
    return IsolationForest(n_estimators=n_estimators, max_samples=max_samples,
                           contamination='auto', random_state=seed)


# ─── Tuning functions ────────────────────────────────────────────────────────

def tune_boosted(X, y, seed=0, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    if HAS_XGB:
        base = XGBClassifier(eval_metric='logloss', random_state=seed)
        grid = {'n_estimators':  [200, 300, 400],
                'max_depth':     [3, 5, 7],
                'learning_rate': [0.03, 0.05, 0.1],
                'subsample':     [0.8, 1.0]}
    else:
        base = HistGradientBoostingClassifier(random_state=seed)
        grid = {'max_iter':      [200, 300, 400],
                'max_depth':     [3, 5, 7],
                'learning_rate': [0.03, 0.05, 0.1]}
    search = GridSearchCV(base, grid, scoring='roc_auc', cv=cv, n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, search.best_score_

def tune_svm_rbf_cv(X, y, seed=0, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('clf', SVC(kernel='rbf', probability=True, random_state=seed))])
    grid = {'clf__C':     [0.1, 1, 10, 100],
            'clf__gamma': ['scale', 0.01, 0.1, 1]}
    search = GridSearchCV(pipe, grid, scoring='roc_auc', cv=cv, n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, search.best_score_

def tune_rf(X, y, seed=0, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    base = RandomForestClassifier(random_state=seed, n_jobs=-1)
    grid = {'n_estimators':     [200, 300, 500],
            'max_depth':        [None, 8, 12],
            'min_samples_leaf': [1, 2, 4]}
    search = GridSearchCV(base, grid, scoring='roc_auc', cv=cv, n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, search.best_score_


print("Model factory and tuning functions defined.")

---
## Section 6 — Hyperparameter Tuning

We run 5-fold stratified cross-validation for the three most powerful models.

> ⏱️ **This is the slowest step** — expect 5–15 minutes depending on hardware and whether XGBoost is installed.


In [ ]:
print("=" * 60)
print("Tuning Boosted Trees — BB84")
print("=" * 60)
boosted84_model, boosted84_params, boosted84_cv = tune_boosted(X84tr, y84tr)
print(f"  Best CV-AUC : {boosted84_cv:.4f}")
print(f"  Best params : {boosted84_params}")

print("\n" + "=" * 60)
print("Tuning SVM-RBF — BB84")
print("=" * 60)
svm84_model, svm84_params, svm84_cv = tune_svm_rbf_cv(X84tr, y84tr)
print(f"  Best CV-AUC : {svm84_cv:.4f}")
print(f"  Best params : {svm84_params}")

print("\n" + "=" * 60)
print("Tuning Random Forest — BB84")
print("=" * 60)
rf84_model, rf84_params, rf84_cv = tune_rf(X84tr, y84tr)
print(f"  Best CV-AUC : {rf84_cv:.4f}")
print(f"  Best params : {rf84_params}")

print("\n" + "=" * 60)
print("Tuning Boosted Trees — BKM07")
print("=" * 60)
boostedbk_model, boostedbk_params, boostedbk_cv = tune_boosted(Xbktr, ybktr)
print(f"  Best CV-AUC : {boostedbk_cv:.4f}")
print(f"  Best params : {boostedbk_params}")
print("\n" + "=" * 60)
print("Tuning Random Forest — BKM07")
print("=" * 60)
rfbk_model, rfbk_params, rfbk_cv = tune_rf(Xbktr, ybktr)
print(f"  Best CV-AUC : {rfbk_cv:.4f}")
print(f"  Best params : {rfbk_params}")

print("\n" + "=" * 60)
print("Tuning SVM-RBF — BKM07")
print("=" * 60)
svmbk_model, svmbk_params, svmbk_cv = tune_svm_rbf_cv(Xbktr, ybktr)
print(f"  Best CV-AUC : {svmbk_cv:.4f}")
print(f"  Best params : {svmbk_params}")

---
## Section 7 — Training the Remaining Models

KNN, Logistic Regression, and the Isolation Forest use fixed settings and are trained in seconds.

The tuned models (Boosted Trees, SVM-RBF, Random Forest) already have their best estimators fitted to the full training set via `best_estimator_` from `GridSearchCV`.


In [ ]:
# ── Supervised models with fixed settings ────────────────────────────────────
print("Fitting KNN and Logistic Regression for BB84 and BKM07 …")
knn84 = make_knn(); knn84.fit(X84tr, y84tr)
lr84  = make_logreg(); lr84.fit(X84tr, y84tr)

knnbk = make_knn(); knnbk.fit(Xbktr, ybktr)
lrbk  = make_logreg(); lrbk.fit(Xbktr, ybktr)

# ── Isolation Forest (unsupervised: trained only on clean data) ───────────────
print("Fitting Isolation Forests …")
clean84 = X84tr[y84tr == 0]   # only secure examples!
cleanbk = Xbktr[ybktr == 0]
ifo84 = make_isolation_forest(); ifo84.fit(clean84)
ifobk = make_isolation_forest(); ifobk.fit(cleanbk)

# Tuned models are already fit — just note their names.
boosted_name = 'XGBoost' if HAS_XGB else 'HistGradientBoosting'
print(f"\nAll models ready:")
print(f"  BB84  : KNN, LogReg, RF (tuned), SVM-RBF (tuned), {boosted_name} (tuned), IsoForest")
print(f"  BKM07 : KNN, LogReg, RF (tuned), SVM-RBF (tuned), {boosted_name} (tuned), IsoForest")

---
## Section 8 — Evaluation on Held-Out Test Set

### Metrics

| Metric | Meaning |
|---|---|
| **ACC** | Fraction of test examples classified correctly at threshold 0.5 |
| **AUC** | Area under the ROC curve — probability that the model ranks a random attacked sample higher than a random secure sample. 1.0 = perfect, 0.5 = random guessing |
| **F1** | Harmonic mean of precision and recall at threshold 0.5 |

### Why report all three?

ACC alone is misleading if class balance is unequal. AUC captures ranking quality across all thresholds. F1 balances false alarms (false positives) against missed detections (false negatives).


In [ ]:
def evaluate(name, model, X_te, y_te):
    '''Evaluate a model and print ACC, AUC, F1. Returns predicted probabilities.'''
    proba = model.predict_proba(X_te)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    acc   = accuracy_score(y_te, pred)
    auc   = roc_auc_score(y_te, proba)
    f1    = f1_score(y_te, pred)
    print(f"  {name:<30}  ACC={acc*100:5.1f}%   AUC={auc:.4f}   F1={f1:.4f}")
    return proba, acc, auc, f1

def isolation_anomaly_scores(model, X_te):
    '''Convert Isolation Forest scores to [0,1] where 1 = most anomalous.'''
    raw = -model.score_samples(X_te)    # flip: high = anomalous
    return (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)


print("━" * 60)
print("BB84 — Test-set performance")
print("━" * 60)
p84_knn, *_ = evaluate("KNN", knn84, X84te, y84te)
p84_lr,  *_ = evaluate("Logistic Regression", lr84, X84te, y84te)
p84_rf,  *_ = evaluate("Random Forest (tuned)", rf84_model, X84te, y84te)
p84_svm, *_ = evaluate("SVM-RBF (tuned)", svm84_model, X84te, y84te)
p84_xgb, *_ = evaluate(f"{boosted_name} (tuned)", boosted84_model, X84te, y84te)
p84_ifo      = isolation_anomaly_scores(ifo84, X84te)
auc84_ifo    = roc_auc_score(y84te, p84_ifo)
print(f"  {'Isolation Forest (unsupervised)':<30}  ACC= N/A    AUC={auc84_ifo:.4f}")

print()
print("━" * 60)
print("BKM07 — Test-set performance")
print("━" * 60)
pbk_knn, *_ = evaluate("KNN", knnbk, Xbkte, ybkte)
pbk_lr,  *_ = evaluate("Logistic Regression", lrbk, Xbkte, ybkte)
pbk_rf,  *_ = evaluate("Random Forest (tuned)", rfbk_model, Xbkte, ybkte)
pbk_svm, *_ = evaluate("SVM-RBF (tuned)", svmbk_model, Xbkte, ybkte)
pbk_xgb, *_ = evaluate(f"{boosted_name} (tuned)", boostedbk_model, Xbkte, ybkte)
pbk_ifo      = isolation_anomaly_scores(ifobk, Xbkte)
aucbk_ifo    = roc_auc_score(ybkte, pbk_ifo)
print(f"  {'Isolation Forest (unsupervised)':<30}  ACC= N/A    AUC={aucbk_ifo:.4f}")

### Model A / A+ / B / C ablation (P14)

Each step adds one kind of information and nothing else, so a change in
performance is attributable to the group that was added:

| Step | Features | Hypothesis under test |
|---|---|---|
| A (QBER only) | `qber_total` | "A conventional single-threshold QBER test" -- the real baseline (what a deployed system does today) |
| A+ (aggregate) | + `qber_z`, `qber_x`, `gain_mu`, `sifted_rate` | "All aggregate protocol statistics" |
| B (+ temporal) | + `qber_dispersion`, `jump_energy_norm`, `spectral_entropy`, `autocorr_lag1` | "Temporal structure adds information beyond aggregates" |
| C (+ decoy, all) | + `y1_lower`, `e1_upper`, `r_secure`, `gain_ratio_nu_mu`, `h_qber` | "Decoy observables add information -- specifically about PNS" |

Per-class FNR (not just AUC) is reported at a fixed 1% false-positive
budget on the validation fold, mean ± 95% CI over `N_REPEATS` reshuffled
splits (P13). Expect `C` to separate `pns` sharply from `A`/`A+`/`B` (the
decoy estimators are the only thing that can see it -- audit Sec. I.2) and
`B` to help specifically on bursty `intercept_resend` (Sec. Defect 3 / P4).
If that pattern is absent, something in the decoy or temporal-profile
plumbing regressed.

In [ ]:
FEATURE_GROUPS_BB84 = {
    'A QBER only':      ['qber_total'],
    'A+ aggregate':      ['qber_total', 'qber_z', 'qber_x', 'gain_mu', 'sifted_rate'],
    'B + temporal':      ['qber_total', 'qber_z', 'qber_x', 'gain_mu', 'sifted_rate',
                           'qber_dispersion', 'jump_energy_norm', 'spectral_entropy', 'autocorr_lag1'],
    'C + decoy (all)':   BB84_FEATURE_NAMES,
}


def run_ablation(X_full, y, feature_names, groups, attack_labels=None,
                  n_repeats=N_REPEATS, seeds=SEEDS):
    """P14: Model A/A+/B/C comparison with per-class FNR and 95% CIs over seeds."""
    idx = {n: i for i, n in enumerate(feature_names)}
    rows = []
    for gname, feats in groups.items():
        cols = [idx[f] for f in feats if f in idx]
        aucs, prs, fnrs = [], [], []
        per_class = {}
        for r in range(n_repeats):
            rs = np.random.RandomState(seeds.master + r)
            perm = rs.permutation(len(y))
            sp = int(0.8 * len(y))
            tr, te = perm[:sp], perm[sp:]
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                continue
            m = make_boosted(seed=r)
            m.fit(X_full[tr][:, cols], y[tr])
            p = m.predict_proba(X_full[te][:, cols])[:, 1]
            aucs.append(roc_auc_score(y[te], p))
            prs.append(average_precision_score(y[te], p))
            neg = p[y[te] == 0]
            thr = np.quantile(neg, 0.99) if len(neg) else 0.5
            fnrs.append(float(np.mean(p[y[te] == 1] <= thr)) if (y[te] == 1).any() else np.nan)
            if attack_labels is not None:
                labels_te = attack_labels[te]
                for cls in np.unique(labels_te):
                    if cls == 'none':
                        continue
                    mm = labels_te == cls
                    if mm.any():
                        per_class.setdefault(cls, []).append(float(np.mean(p[mm] <= thr)))
        a = mean_ci(aucs)
        pr = mean_ci(prs)
        fn = mean_ci(fnrs)
        row = dict(model=gname, n_feat=len(cols),
                   auc=f"{a[0]:.4f} [{a[1]:.4f},{a[2]:.4f}]",
                   pr_auc=f"{pr[0]:.4f} [{pr[1]:.4f},{pr[2]:.4f}]",
                   fnr_at_1pct_fpr=f"{fn[0]:.3f} [{fn[1]:.3f},{fn[2]:.3f}]")
        for cls, vals in per_class.items():
            c = mean_ci(vals)
            row[f'FNR_{cls}'] = f"{c[0]:.3f}"
        rows.append(row)
    return pd.DataFrame(rows)


print(f"Running the Model A/A+/B/C ablation for BB84 ({N_REPEATS} repeats per group)...")
abl_bb84 = run_ablation(bb84_X, bb84_y, BB84_FEATURE_NAMES, FEATURE_GROUPS_BB84,
                         attack_labels=bb84_attack_labels)
print()
print(abl_bb84.to_string(index=False))
print()
print("Expect FNR_pns to collapse specifically at step C (decoy observables) and")
print("FNR_intercept_resend to improve specifically at step B (temporal features) --")
print("that pattern is also a correctness check on the physics, not just an ML result.")

---
## Section 9 — Generalisation Testing (E1-E4)

A model that only performs well on the same distribution it was trained on is not evidence it learned an attack *signature* — it could just as easily have learned "this specific combination of channel conditions." Four splits, in increasing order of how hard they are to pass:

| Split | What it tests |
|---|---|
| **E1** (reference) | The ordinary random 80/20 split already reported in Section 8 — the *weakest* evidence, since train and test share the same distribution of everything. |
| **E2** | Train on weak attack intensity, test on strong (and the harder reverse direction) — does the model generalise across *how hard* Eve is attacking? |
| **E3** | Train on short-distance links, test on long-distance (and reverse) — a naive QBER-threshold detector fails here, since distance-induced degradation can look like an attack. |
| **E4** | Leave-one-cell-out over a (distance bin x intensity bin) grid — the strictest test: every training run excludes an entire region of channel-condition space that the test run sits in. |

Each split reports AUC and FNR at a fixed 1% false-positive budget, not just accuracy — and E4 additionally reports a mean AUC with a 95% CI across cells, since any single held-out cell can be small. A wide CI here is itself informative: it says the dataset scale (Section 4) isn't yet large enough to pin down cross-cell generalisation tightly, which is different from (and more honest than) claiming a single point estimate.

(BKM07's distance range is only 0-15km for round-trip-loss reasons, so E2-E4 are reported for BB84 only, which has the full 0-100km range to split on.)

In [ ]:
def grouped_eval(X, y, group_values, train_mask, test_mask, seed=0, name=''):
    """Train on one region of a nuisance variable, test on a disjoint region."""
    if train_mask.sum() < 10 or test_mask.sum() < 10:
        print(f"  {name}: insufficient samples "
              f"({train_mask.sum()} train / {test_mask.sum()} test) -- skipped")
        return None
    m = make_boosted(seed=seed)
    m.fit(X[train_mask], y[train_mask])
    p = m.predict_proba(X[test_mask])[:, 1]
    yte = y[test_mask]
    if len(np.unique(yte)) < 2:
        print(f"  {name}: test set has a single class -- skipped")
        return None
    auc = roc_auc_score(yte, p)
    neg = p[yte == 0]
    thr = np.quantile(neg, 0.99) if len(neg) else 0.5
    fnr = float(np.mean(p[yte == 1] <= thr)) if (yte == 1).any() else np.nan
    print(f"  {name:<46} n_tr={train_mask.sum():5d} n_te={test_mask.sum():5d} "
          f"AUC={auc:.4f} FNR@1%FPR={fnr:.3f}")
    return auc, fnr


print("E1 Random stratified split (in-distribution) -- see the BB84/BKM07")
print("   'Test-set performance' tables above. This is the WEAKEST evidence")
print("   and should not be the headline result.")

print("\nE2 Unseen attack intensity (BB84)")
sec = bb84_y == 0
weak = bb84_eve_intensity_col <= 0.10
strong = bb84_eve_intensity_col >= 0.30
grouped_eval(bb84_X, bb84_y, bb84_eve_intensity_col, sec | weak, sec | strong,
             name='train weak (<=0.10) -> test strong (>=0.30)')
grouped_eval(bb84_X, bb84_y, bb84_eve_intensity_col, sec | strong, sec | weak,
             name='train strong (>=0.30) -> test weak (<=0.10) [hard direction]')

print("\nE3 Unseen distance range (BB84)")
near = bb84_noise <= 40
far = bb84_noise >= 60
grouped_eval(bb84_X, bb84_y, bb84_noise, near, far, name='train 0-40 km -> test 60-100 km')
grouped_eval(bb84_X, bb84_y, bb84_noise, far, near, name='train 60-100 km -> test 0-40 km')
print("  (This is where a naive QBER-threshold detector would fail: distance-induced")
print("   degradation can look like an attack. Compare against E1's in-distribution AUC.)")

print("\nE4 Leave-one-cell-out, BB84, over a (distance bin x intensity bin) grid")
dbin = np.digitize(bb84_noise, [20, 40, 60, 80])
ibin = np.digitize(bb84_eve_intensity_col, [0.15, 0.35, 0.6])
cell_id = dbin * 10 + ibin
_e4_aucs = []
for _c in np.unique(cell_id):
    te = cell_id == _c
    tr = ~te
    r = grouped_eval(bb84_X, bb84_y, cell_id, tr, te, name=f'hold out cell {_c}')
    if r is not None:
        _e4_aucs.append(r[0])
if _e4_aucs:
    _m, _lo, _hi = mean_ci(_e4_aucs)
    print(f"\n  LOCO mean AUC = {_m:.4f}  95% CI [{_lo:.4f}, {_hi:.4f}]  over {len(_e4_aucs)} cells")
else:
    print("\n  LOCO: no cell had >=10 samples on both sides -- skipped (dataset too small; "
          "increase samples_per_class in Section 4's generate_datasets() call for a real run)")

print("\n(BKM07's distance range was narrowed to 0-15km for the round-trip-loss reasons")
print(" in Section 9/P10, so E2-E4 are reported for BB84 only, which has the full 0-100km range.)")

---
## Section 10 — ROC Curve Comparison

The **Receiver Operating Characteristic (ROC) curve** plots the true positive rate (TPR = correctly identified attacks) against the false positive rate (FPR = false alarms) as we vary the classification threshold.

- A perfect classifier hugs the top-left corner (AUC = 1.0).
- The dashed diagonal is random guessing (AUC = 0.5).

The shaded region under each curve is its AUC, shown in the legend.


In [ ]:
matplotlib.use('Agg')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bb84_curves = [
    ('KNN',                  p84_knn, '#94A3B8'),
    ('Logistic Regression',  p84_lr,  '#0EA5E9'),
    ('Random Forest',        p84_rf,  '#6366F1'),
    ('SVM-RBF',              p84_svm, '#F59E0B'),
    (boosted_name,           p84_xgb, '#DC2626'),
    ('Isolation Forest',     p84_ifo, '#16A34A'),
]
bkm_curves = [
    ('KNN',                  pbk_knn, '#94A3B8'),
    ('Logistic Regression',  pbk_lr,  '#0EA5E9'),
    ('Random Forest',        pbk_rf,  '#6366F1'),
    ('SVM-RBF',              pbk_svm, '#F59E0B'),
    (boosted_name,           pbk_xgb, '#DC2626'),
    ('Isolation Forest',     pbk_ifo, '#16A34A'),
]

for ax, y, curves, title in [
        (axes[0], y84te, bb84_curves, 'BB84 (Fully Quantum)'),
        (axes[1], ybkte, bkm_curves,  'BKM07 (Semi-Quantum)')]:
    for label, proba, c in curves:
        fpr, tpr, _ = roc_curve(y, proba)
        auc_v = roc_auc_score(y, proba)
        ax.fill_between(fpr, tpr, alpha=0.06, color=c)
        ax.plot(fpr, tpr, color=c, linewidth=2.0,
                label=f'{label}  (AUC = {auc_v:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', linewidth=1.2, label='Random guessing')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8.5, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])

fig.suptitle('ROC Curves: Eavesdropping Detection — BB84 vs BKM07\n'
             '(Fully Quantum vs Semi-Quantum Protocol)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/roc_comparison_v2.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: plots/roc_comparison_v2.png")

---
## Section 11 — Feature Importance

Tree-based models offer a built-in *impurity* importance (how much does splitting on this feature reduce loss, averaged across trees) — but that measure is biased toward continuous / high-cardinality features (Strobl, Boulesteix, Zeileis & Hothorn, BMC Bioinformatics 8:25, 2007), which matters here because several of our features are near-collinear by construction (`qber_total`/`qber_z`/`qber_x`/`h_qber`; `qber_zs`/`qber_key` in BKM07).

Instead we use **permutation importance** (`sklearn.inspection.permutation_importance`): shuffle one feature column on the held-out test set and measure the resulting drop in ROC-AUC. This measures each feature's actual contribution to the *model's real predictive performance*, not an internal training-time bookkeeping statistic — at the cost of being itself biased under feature correlation (Hooker, Mentch & Zhou, Stat. Comput. 31:82, 2021), so the plot should be read as a ranking among *groups* of correlated features, not a precise per-column attribution.

We plot these for the tuned **Random Forest** on both BB84 and BKM07.

Key things to look for:
- If `qber_total` (BB84) or `qber_key` (BKM07) dominates → the model mostly relies on average error level, which Section 8b's ablation (Model A/A+/B/C) already quantifies directly.
- If `y1_lower`/`e1_upper`/`gain_ratio_nu_mu` (BB84's decoy-state block) rank high → the model is using PNS-specific information a naive QBER threshold could never see.
- If `jump_energy_norm`/`qber_dispersion`/`spectral_entropy` rank high → temporal structure is doing real work, consistent with Section 8b's finding that the temporal feature group helps specifically on bursty intercept-resend.
- If `asymmetry` is high for BKM07 → the model uses the forward/return leg imbalance.

In [ ]:
bb84_feat_names = BB84_FEATURE_NAMES
bkm_feat_names = BKM_FEATURE_NAMES

# P18: permutation importance on the HELD-OUT test set, not impurity
# importance. Impurity-based importance is biased toward continuous /
# high-cardinality features (Strobl, Boulesteix, Zeileis & Hothorn, BMC
# Bioinformatics 8:25, 2007), and several of our features are near-collinear
# by construction (qber_total/qber_z/qber_x/h_qber; delta_/corr_ pairs in
# E91) -- exactly the situation impurity importance handles worst.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (model, names, Xte, yte, title, colour) in zip(axes, [
        (rf84_model, bb84_feat_names, X84te, y84te,
         'BB84 — Random Forest Permutation Importance', '#2563EB'),
        (rfbk_model, bkm_feat_names, Xbkte, ybkte,
         'BKM07 — Random Forest Permutation Importance', '#DC2626'),
]):
    r = permutation_importance(model, Xte, yte, n_repeats=30, random_state=0,
                                scoring='roc_auc', n_jobs=-1)
    imp, imp_sd = r.importances_mean, r.importances_std
    scale = imp.max() if imp.max() > 0 else 1.0
    imp_n = imp / scale
    sd_n = imp_sd / scale

    order = np.argsort(imp_n)
    sorted_names = [names[i] for i in order]
    sorted_imp = imp_n[order]
    sorted_sd = sd_n[order]

    ax.barh(sorted_names, sorted_imp, xerr=sorted_sd, color=colour, alpha=0.82, capsize=2)
    ax.set_xlabel('Normalised permutation importance (drop in held-out ROC-AUC)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Feature Importance: Which Signal Reveals Eve? (P18: permutation importance)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_importance_v2.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: plots/feature_importance_v2.png")
print()
print("Note (Hooker, Mentch & Zhou, Stat. Comput. 31:82, 2021): permutation")
print("importance is itself biased under feature correlation -- read the plot as a")
print("ranking among GROUPS of correlated features, not a precise attribution to")
print("any single column.")

### N x window sensitivity (P17)

How many pulses per run, and how many sub-windows, do the temporal
features (`qber_dispersion`, `jump_energy_norm`, `spectral_entropy`,
`autocorr_lag1`) actually need before they carry real information rather
than estimator noise? Per the audit (Sec. H.1): the per-window QBER
estimator needs `n_w >= 100` sifted bits for its standard error to stay
below the quantity being estimated, and `spectral_entropy`/`autocorr_lag1`
need `W >= 50-64` windows before their own estimator bias/variance settles
down. Below computes `AUC_A` (QBER-total alone) and `AUC_B` (QBER +
aggregate + temporal) across a grid of `(N pulses, W windows)` and plots
`ΔAUC = AUC_B - AUC_A` -- the value the temporal block actually adds, as a
function of the statistics available to compute it.

In [ ]:
def n_window_sensitivity(N_values=(200_000, 1_000_000, 2_000_000),
                          W_values=(16, 32, 64),
                          n_per_class=8, distance_km=25.0,
                          eve_intensity=0.15, profile='bursty'):
    """P17: how many pulses N and how many windows W do the temporal
    features need? Reports AUC_A (qber_total alone) vs AUC_B (+ aggregate +
    temporal), and n_w (mean sifted bits per window -- the binomial-noise
    driver, audit Sec. H.1)."""
    A_FEATS = ['qber_total']
    B_FEATS = ['qber_total', 'gain_mu', 'sifted_rate', 'qber_dispersion',
               'jump_energy_norm', 'spectral_entropy', 'autocorr_lag1']
    idx = {n: i for i, n in enumerate(BB84_FEATURE_NAMES)}
    out = []
    for N in N_values:
        for W in W_values:
            rows, ys, nws = [], [], []
            for cls, inten in [('none', 0.0), ('intercept_resend', eve_intensity)]:
                for i in range(n_per_class):
                    rng = SEEDS.rng('nw_exp', i)
                    f = collect_bb84_features(int(N), distance_km, eve_mode=cls,
                                               eve_intensity=inten, profile=profile,
                                               n_windows=W, rng=rng)
                    rows.append([f[k] for k in BB84_FEATURE_NAMES])
                    ys.append(0 if cls == 'none' else 1)
                    nws.append(f['sifted_rate'] * N / W)
            X = np.array(rows, dtype=float)
            y = np.array(ys)
            ok = np.isfinite(X).all(axis=1)
            if ok.sum() < 8 or len(np.unique(y[ok])) < 2:
                out.append(dict(N=N, W=W, n_w=float(np.nanmean(nws)), AUC_A=np.nan, AUC_B=np.nan, dAUC=np.nan))
                print(f"  N={N:>9,} W={W:>4}: insufficient data -- skipped")
                continue
            X, y = X[ok], y[ok]
            perm = np.random.RandomState(0).permutation(len(y))
            sp = max(1, int(0.7 * len(y)))
            res = {}
            for tag, feats in [('A', A_FEATS), ('B', B_FEATS)]:
                cols = [idx[f] for f in feats]
                if len(np.unique(y[perm[:sp]])) < 2 or len(np.unique(y[perm[sp:]])) < 2:
                    res[tag] = np.nan
                    continue
                m = make_boosted(seed=0)
                m.fit(X[perm[:sp]][:, cols], y[perm[:sp]])
                res[tag] = roc_auc_score(y[perm[sp:]], m.predict_proba(X[perm[sp:]][:, cols])[:, 1])
            dauc = (res['B'] - res['A']) if np.isfinite(res.get('A', np.nan)) and np.isfinite(res.get('B', np.nan)) else np.nan
            out.append(dict(N=N, W=W, n_w=float(np.mean(nws)), AUC_A=res.get('A', np.nan),
                             AUC_B=res.get('B', np.nan), dAUC=dauc))
            print(f"  N={N:>9,} W={W:>4} n_w={np.mean(nws):8.1f} "
                  f"AUC_A={res.get('A', float('nan')):.4f} AUC_B={res.get('B', float('nan')):.4f} "
                  f"dAUC={dauc:+.4f}" if np.isfinite(dauc) else
                  f"  N={N:>9,} W={W:>4} n_w={np.mean(nws):8.1f} (one split had a single class)")
    return pd.DataFrame(out)


print("Running the N x window sensitivity grid (scaled down for runtime -- see the")
print("markdown note above)...")
nw_df = n_window_sensitivity()
print()
print(nw_df.round(4).to_string(index=False))

piv = nw_df.pivot(index='N', columns='W', values='dAUC')
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(piv.values, aspect='auto', origin='lower', cmap='RdYlGn', vmin=-0.2, vmax=0.2)
ax.set_xticks(range(len(piv.columns)))
ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index)))
ax.set_yticklabels([f"{int(v):,}" for v in piv.index])
ax.set_xlabel('number of windows W')
ax.set_ylabel('pulses per run N')
ax.set_title('Value added by temporal features: AUC(B) - AUC(A)')
for _i in range(piv.shape[0]):
    for _j in range(piv.shape[1]):
        _v = piv.values[_i, _j]
        if np.isfinite(_v):
            ax.text(_j, _i, f"{_v:+.3f}", ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Delta AUC (B - A)')
plt.tight_layout()
plt.savefig('plots/n_window_sensitivity.png', dpi=200)
plt.show()
print("Saved: plots/n_window_sensitivity.png")

---
## Section 12 — Summary & Conclusions

### What this notebook does

1. Simulates BB84 and BKM07 on a physically realistic fibre-link channel model (GYS-calibrated loss, dark counts, detector misalignment — Section 1-2), and E91 as a genuine two-qubit density-matrix simulation with CPTP-map attacks (Section 13).
2. Engineers a 14-feature vector per run for each protocol, spanning aggregate error rates, decoy-state PNS estimators (BB84), and temporal-structure descriptors (Section 3).
3. Runs an 8-test leakage audit (Kapoor & Narayanan, Patterns 4, 100804, 2023) on every generated dataset *before* training any model, so a clean audit is a precondition for trusting the numbers below, not an afterthought.
4. Trains and compares six classifiers per protocol, three hyperparameter-tuned via 5-fold stratified cross-validation optimising AUC (Section 5-8).
5. Quantifies *which* feature group drives detection via a nested ablation (Model A/A+/B/C, Section 8b) and *why* via permutation importance (Section 11) — not impurity importance, which is biased toward this feature set's collinear structure.
6. Tests generalisation across four distribution shifts (Section 9), and how much the temporal-feature block's contribution depends on run length / window count (Section 8c).
7. Calibrates all three protocols to a shared target honest QBER (Section 3's matched-QBER layer) so a future cross-protocol comparison isn't confounded by each protocol's knob mapping to QBER differently.

### Key findings (see each section's own printed numbers for the current run)

- Detection is easy when the relevant feature group is present and the attack is strong, and gets substantially harder as the ablation and generalisation sections isolate *which* information source is actually doing the work — the honest headline number is the ablation/generalisation result, not the in-distribution test-set AUC alone (Section 8b/9 make this explicit).
- PNS is close to invisible without the decoy-state estimators (`y1_lower`/`e1_upper`/`gain_ratio_nu_mu`) — a conventional QBER-threshold detector would miss it almost entirely.
- Isolation Forest (unsupervised, trained only on secure data) achieves meaningfully-above-chance AUC — useful when labelled attack data is unavailable, though weaker than the supervised models.

### Known limitations

| Limitation | What it means in practice |
|---|---|
| Individual (per-pulse) attacks only | Coherent/collective attacks (Eve stores qubits, measures later using information revealed in sifting) are strictly stronger and not modelled — a real security proof needs to account for them. |
| No finite-key composable security | `r_secure` (Section 3) is an *asymptotic* GLLP+decoy rate used as a channel-state feature, not a finite-key security bound (Tomamichel et al. 2012 develops the composable version). |
| No detector-side attacks | Detector blinding / efficiency-mismatch attacks (Lydersen et al. 2010) act on the physical detector, not the protocol's bit/correlation stream — invisible to a channel model like this one by construction. |
| Dataset scale is reduced for runtime | `samples_per_class=60` keeps the notebook runnable in a few minutes; Section 9's leave-one-cell-out confidence intervals are visibly wide at this scale — a real study should increase it. |
| E91's attack set omits detector/source-hardware attacks | The original `g2_0`/`detection_efficiency`/`detector_curve_residual`/`photocurrent_baseline` features were removed (Section 13) because they were label-conditioned constructs, not genuinely simulated — a real PNS/blinding analogue for E91 needs an actual source/detector model, which is out of scope here. |

### Suggested next steps

- Extend `channel_model()` with a loss-dependent sifted-key *rate* forecast, not just QBER, for a more complete link budget.
- Extend to multi-class classification to distinguish attack *types* (not just secure-vs-attacked) in one model.
- Try conformal prediction for calibrated per-classification confidence intervals.
- Use the Section 3 calibration layer to run an actual matched-QBER cross-protocol AUC comparison (the layer is built; the comparison grid itself is not run here).
- Test on real QBER traces from a hardware QKD testbed.

# E91

## Section 13 — E91: Entanglement-Based QKD

Unlike BB84 (Alice sends prepared photons) and BKM07 (Bob is classical),
**E91** distributes entangled photon pairs from a shared source to Alice
and Bob. As of this patch, the honest channel is a genuine **4x4 density
matrix** (a Werner state with visibility `V`) and every attack is a
genuine **CPTP map** on that state (P7/P8), rather than a table lookup on
the analytic singlet correlation — this is what lets the honest channel
have real, non-adversarial noise (`V < 1`) and what fixes a real bug in
the previous version: mismatched `sin²`/`cos²` conventions on the
Alice-Eve vs Eve-Bob legs, plus resampling Eve's basis independently *per
CHSH correlator* instead of once per pair, which together pushed the
measured `|S|` for 100% intercept-resend to 2.77 instead of the correct
~1.39–1.41 (a local-hidden-variable channel cannot violate CHSH at all).

### Measurement convention
Angles use Ekert's original **polarisation** convention,
`E(θ_a,θ_b) = -cos(2·(θ_a-θ_b))`, with his optimal 22.5°-spaced settings.

- Alice picks from `{a1=0°, a2=22.5°, a3=45°}`; Bob from `{b1=22.5°, b2=45°, b3=67.5°}`.
- `a2==b1` and `a3==b2` are angle-matched → the ideal correlation there is
  `E=-1` (perfect **anti**-correlation) → these generate the raw key.
- CHSH uses `a1,a3` against `b1,b3`: **S = E(a1,b1) − E(a1,b3) + E(a3,b1) + E(a3,b3)**.
  Classical bound `|S|≤2`; quantum (Tsirelson) bound `|S|≤2√2≈2.828`.

### Attack set (revised — see the note at the end of this section)
The previous version's `pns` and `detector_blind` E91 classes were **not**
simulated attacks: their tell-tale features (`g2_0`,
`detection_efficiency`, `detector_curve_residual`, `photocurrent_baseline`)
were sampled directly from a distribution *chosen by the ground-truth
label* — e.g. `detection_efficiency ~ N(0.70,0.05)` for every non-blind
class and `N(0.15,0.03)` for `detector_blind`, with **zero overlap**. A
single threshold on that feature alone gives AUC≈1.0 by construction; that
is not eavesdropping detection, it is detecting which branch of an
`if`-statement ran. Those four features and the two classes they existed
to support are removed.

| Attack | Mechanism | Visible in `chsh_S`/`qber_key`? |
|---|---|---|
| `intercept_resend` | Measure-and-resend on Bob's arm, basis-averaged CPTP map | Yes — QBER rises, `|S|` collapses toward the classical bound |
| `ancilla` | Entangling probe on Bob's arm (basis-anisotropic dephasing) | Partially — costs *more* QBER than CHSH, which the new `s_qber_residual` feature is built to catch |
| `loss_manipulation` | Asymmetric arm attenuation | Similar to honest depolarisation at the aggregate level |

This is also the direct test of whether physics-informed features add
anything beyond a conventional Bell test: Section 14's "critical baseline"
compares a model trained on `chsh_S` alone against the full feature set on
the *same* labelled data.

## Section 14 — E91 Eavesdropping Detection

Generate a labelled dataset with `generate_e91_dataset()` (4 classes:
`none`, `intercept_resend`, `ancilla`, `loss_manipulation` -- Section 13),
split train/test with the same `RandomState(7)` convention as BB84/BKM07,
train the same model families (KNN, Logistic Regression, Random Forest,
Boosted trees) plus the unsupervised Isolation Forest, and evaluate.

Two things this section adds that BB84/BKM07 don't need:

1. **FNR-focused reporting**, not just accuracy/AUC -- an attacked sample
   (of any of the three attack types) classified as secure is the actual
   security failure, and accuracy alone can hide a bad false-negative rate
   on a balanced dataset.
2. **The critical baseline**: `chsh_S` alone (what a conventional Bell test
   reduces to) versus the full 12-feature physics-informed set, on the
   *same* model. Per Section 13's note, basis-averaged `intercept_resend`
   sits on the same honest-depolarisation curve as ordinary noise in
   `chsh_S`/`qber_key` alone, so the CHSH-only model is expected to do
   worst specifically on that class; `ancilla` is the case
   `s_qber_residual` exists to catch. Expect a SMALL, honest gap between
   the two models here -- a large gap would suggest something upstream is
   leaking rather than the physics-informed features genuinely helping.

In [ ]:
E91_FEATURE_NAMES = ['qber_key', 'chsh_S', 's_deviation', 's_qber_residual',
                      'per_pair_corr_spread', 'jump_energy', 'spectral_entropy',
                      'autocorr_lag1', 'h_qber_key', 'sifted_rate']
E91_CHSH_ONLY_NAMES = ['chsh_S']

print("Building E91 dataset (4 classes: none / intercept_resend / ancilla / loss_manipulation) ...")
e91_df = generate_e91_dataset(n_runs_per_mode=100, n_pulses=5000, n_windows=32,
                               eve_modes=("none", "intercept_resend", "ancilla", "loss_manipulation"))
e91_df.to_csv('data/e91_dataset.csv', index=False)

X91 = e91_df[E91_FEATURE_NAMES].to_numpy(dtype=float)
label91 = e91_df['label'].to_numpy()
duty91 = e91_df['attack_duty_cycle'].to_numpy(dtype=float)
y91 = (label91 != 'none').astype(int)

print(f"\nE91 dataset shape: {X91.shape}   (label counts: {dict(zip(*np.unique(label91, return_counts=True)))})")
print("CSV saved to data/e91_dataset.csv")

# ── Train/test split (same RandomState(7) convention as BB84/BKM07) ─────────
rng91 = np.random.RandomState(7)
idx91 = rng91.permutation(len(X91))
sp91 = int(0.8 * len(X91))
tr91, te91 = idx91[:sp91], idx91[sp91:]

def e91_cols(names):
    return [E91_FEATURE_NAMES.index(n) for n in names]

Xe91tr,      Xe91te      = X91[tr91],                                  X91[te91]
Xe91tr_chsh, Xe91te_chsh = X91[tr91][:, e91_cols(E91_CHSH_ONLY_NAMES)], X91[te91][:, e91_cols(E91_CHSH_ONLY_NAMES)]
ye91tr,      ye91te      = y91[tr91],                                  y91[te91]
label91_tr,  label91_te  = label91[tr91],                              label91[te91]
duty91_tr,   duty91_te   = duty91[tr91],                               duty91[te91]

print(f"E91 -- train: {Xe91tr.shape}, test: {Xe91te.shape}")
print(f"E91 class balance in train: {dict(zip(*np.unique(ye91tr, return_counts=True)))}")
print(f"E91 test-set attack-type breakdown: {dict(zip(*np.unique(label91_te, return_counts=True)))}")

# ── P16 leakage audit for E91 (V is the hidden nuisance parameter) ─────────
e91_V = e91_df['V'].to_numpy(dtype=float)
leakage_audit(X91, y91, E91_FEATURE_NAMES, label91,
              {'V': e91_V}, np.full(len(X91), 5000))

In [ ]:
# ── Train models (reuses factories from Section 5) ───────────────────────────
print("Fitting KNN and Logistic Regression for E91 ...")
knn91 = make_knn(); knn91.fit(Xe91tr, ye91tr)
lr91  = make_logreg(); lr91.fit(Xe91tr, ye91tr)

print("Tuning Random Forest for E91 ...")
rf91_model, rf91_params, rf91_cv = tune_rf(Xe91tr, ye91tr)
print(f"  Best CV-AUC : {rf91_cv:.4f}")
print(f"  Best params : {rf91_params}")

print("Tuning Boosted Trees for E91 ...")
boosted91_model, boosted91_params, boosted91_cv = tune_boosted(Xe91tr, ye91tr)
print(f"  Best CV-AUC : {boosted91_cv:.4f}")
print(f"  Best params : {boosted91_params}")

print("Fitting Isolation Forest (trained only on secure E91 samples) ...")
clean91 = Xe91tr[ye91tr == 0]
ifo91 = make_isolation_forest(); ifo91.fit(clean91)

print(f"\nE91 models ready: KNN, LogReg, Random Forest (tuned), {boosted_name} (tuned), IsoForest")


def security_report(name, proba, y_te, threshold=0.5):
    '''Accuracy/AUC (as in evaluate()) PLUS the security-relevant confusion
    counts: FN = an attacked sample wrongly called secure -- the actual
    eavesdropping-detection failure, not just a lower accuracy number.'''
    pred = (proba >= threshold).astype(int)
    tp = int(((pred == 1) & (y_te == 1)).sum()); fn = int(((pred == 0) & (y_te == 1)).sum())
    fp = int(((pred == 1) & (y_te == 0)).sum()); tn = int(((pred == 0) & (y_te == 0)).sum())
    fnr = fn / (tp + fn) if (tp + fn) else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) else float('nan')
    acc = (tp + tn) / len(y_te)
    auc = roc_auc_score(y_te, proba)
    print(f"  {name:<32}ACC={acc*100:5.1f}%  AUC={auc:.4f}  FNR={fnr:.3f} (missed {fn}/{tp+fn})  FPR={fpr:.3f}")
    return fnr, fpr, auc


print("━" * 60)
print("E91 -- Test-set performance (full physics-informed feature set)")
print("━" * 60)
p91_knn = knn91.predict_proba(Xe91te)[:, 1]
p91_lr  = lr91.predict_proba(Xe91te)[:, 1]
p91_rf  = rf91_model.predict_proba(Xe91te)[:, 1]
p91_xgb = boosted91_model.predict_proba(Xe91te)[:, 1]
p91_ifo = isolation_anomaly_scores(ifo91, Xe91te)

security_report("KNN", p91_knn, ye91te)
security_report("Logistic Regression", p91_lr, ye91te)
security_report("Random Forest (tuned)", p91_rf, ye91te)
security_report(f"{boosted_name} (tuned)", p91_xgb, ye91te)
auc91_ifo = roc_auc_score(ye91te, p91_ifo)
print(f"  {'Isolation Forest (unsupervised)':<32}ACC= N/A    AUC={auc91_ifo:.4f}")

In [ ]:
# ── Critical baseline: CHSH alone vs the full physics-informed feature set ──
# Same model (boosted trees), two feature sets: does ML learn more than a
# conventional single-threshold Bell test would?
print("Critical baseline -- CHSH-only vs full physics-informed features (same model)")
print("-" * 70)
boosted91_chsh_only, chsh_only_params, chsh_only_cv = tune_boosted(Xe91tr_chsh, ye91tr)
p91_chsh_only = boosted91_chsh_only.predict_proba(Xe91te_chsh)[:, 1]

fnr_chsh, fpr_chsh, auc_chsh = security_report(
    f"{boosted_name}, chsh_only (1 feature)", p91_chsh_only, ye91te)
fnr_full, fpr_full, auc_full = security_report(
    f"{boosted_name}, full ({len(E91_FEATURE_NAMES)} features)", p91_xgb, ye91te)

print(f"\n  ==> FNR {fnr_chsh:.3f} -> {fnr_full:.3f}   |   AUC {auc_chsh:.3f} -> {auc_full:.3f}")
print("  Per the audit's Sec. F.3 finding: basis-averaged intercept-resend sits on")
print("  the SAME |S| = 2*sqrt(2)*(1-2Q) honest-depolarisation curve as ordinary")
print("  noise, so aggregate CHSH/QBER alone cannot separate it -- only temporal")
print("  structure can. `ancilla` attacks are the exception: they are basis-")
print("  anisotropic (cost more QBER than CHSH), which is exactly what")
print("  s_qber_residual is built to catch. Expect a SMALLER, more honest gap")
print("  here than in the pre-patch version -- that gap was mostly the leak.")

# Break the FNR down by attack type -- the direct evidence for the
# per-attack claim above.
print("\n  FNR by attack type (full-feature model vs chsh-only):")
for mode in ("intercept_resend", "ancilla", "loss_manipulation"):
    m = label91_te == mode
    if m.sum() == 0:
        continue
    fnr_full_mode = float(np.mean(p91_xgb[m] < 0.5))
    fnr_chsh_mode = float(np.mean(p91_chsh_only[m] < 0.5))
    print(f"    {mode:<18} n={m.sum():<4} FNR(full)={fnr_full_mode:.3f}   FNR(chsh_only)={fnr_chsh_mode:.3f}")

# ── Adversarial generalisation (same style as Section 11) ─────────────
# Only intercept_resend has a continuous attack-strength knob (duty cycle);
# ancilla/loss_manipulation strengths are drawn from a different range, so
# this sub-test is restricted to secure-vs-intercept_resend rows.
print("\nAdversarial generalisation: train weak intercept-resend (duty<=0.3),")
print("test strong intercept-resend (duty>=0.6)")
low_duty, high_duty = 0.3, 0.6
mask_lo91 = (label91_tr == 'none') | ((label91_tr == 'intercept_resend') & (duty91_tr <= low_duty))
mask_hi91 = (label91_te == 'none') | ((label91_te == 'intercept_resend') & (duty91_te >= high_duty))
print(f"E91 -- weak-attack train samples : {mask_lo91.sum()}")
print(f"E91 -- strong-attack test samples: {mask_hi91.sum()}")
if mask_lo91.sum() > 10 and mask_hi91.sum() > 10:
    adv91 = make_boosted(seed=1)
    adv91.fit(Xe91tr[mask_lo91], ye91tr[mask_lo91])
    p_adv91 = adv91.predict_proba(Xe91te[mask_hi91])[:, 1]
    fnr_adv, fpr_adv, auc_adv = security_report("E91 boosted (duty-cycle gen.)", p_adv91, ye91te[mask_hi91])
else:
    print("E91 -- not enough samples in one or both subsets; skipping.")

In [ ]:
# ── ROC curve (same style as Section 10) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

e91_curves = [
    ('KNN',                    p91_knn,        '#94A3B8'),
    ('Logistic Regression',    p91_lr,         '#0EA5E9'),
    ('Random Forest',          p91_rf,         '#6366F1'),
    (boosted_name,             p91_xgb,        '#DC2626'),
    (f'{boosted_name} (chsh_only)', p91_chsh_only, '#F59E0B'),
    ('Isolation Forest',       p91_ifo,        '#16A34A'),
]

for label, proba, c in e91_curves:
    fpr, tpr, _ = roc_curve(ye91te, proba)
    auc_v = roc_auc_score(ye91te, proba)
    ax.fill_between(fpr, tpr, alpha=0.06, color=c)
    ax.plot(fpr, tpr, color=c, linewidth=2.0, label=f'{label}  (AUC = {auc_v:.3f})')

ax.plot([0, 1], [0, 1], 'k:', linewidth=1.2, label='Random guessing')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('E91 (Entanglement-Based) -- Eavesdropping Detection', fontsize=13, fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig('plots/roc_e91.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: plots/roc_e91.png")

### Scope note (E91, same as BB84/BKM07 above)

`s_deviation` (the CHSH-vs-Tsirelson-bound gap) is a point estimate under a
simplified, idealised model — not a rigorous statistical-significance test
on finite coincidence counts (no `sigma_S`/z-score is computed here), and
not a device-independent security proof. `s_qber_residual` (deviation from
the honest depolarisation curve `|S|=2√2(1-2Q)`) is a genuine
physics-derived diagnostic for basis-anisotropic disturbance, not a
detector-hardware measurement.

Three attack mechanisms are modelled (intercept-resend on Bob's arm, an
entangling-ancilla probe, and asymmetric arm-loss manipulation), each as a
genuine CPTP map on a real two-qubit density matrix. A real E91 deployment
would also need to consider collective/coherent attacks, detector-side
attacks (blinding, efficiency mismatch — genuinely invisible to a channel
model like this one; see Lydersen et al. 2010), trojan-horse attacks on
the source apparatus, and an authenticated classical channel, all of which
are out of scope here. As with BB84/BKM07, these results describe
detection of statistical signatures consistent with eavesdropping under
the simulated channel/attack model, not a formal QKD security guarantee.

# Deep Learning

## Section 15 -- Cross-Protocol Deep Learning Detection

Everything above (Sections 1-14) uses **hand-engineered features**
(`qber_total`, decoy-state estimators, `s_qber_residual`, ...) fed into
classical classifiers. This section asks a different question: can a
**learned representation** -- a CNN+LSTM+attention encoder reading the raw
per-round click/error/basis stream directly, no hand-engineered features
at all -- detect eavesdropping, and does that representation **transfer
across protocols**?

Two research questions, addressed in order:

1. **Leave-one-protocol-out transfer** (Section 15.7): pretrain a shared
   trunk on two protocols, freeze it, and fine-tune only a small adapter
   on a *small fraction* of the third (target) protocol's data. If that
   beats training a fresh model on the same small fraction, the trunk has
   learned attack signatures that generalise across BB84/BKM07/E91's very
   different physical mechanisms -- not just protocol-specific curve
   fitting.
2. **Does supervised pretraining help or hurt zero-day detection?**
   (Section 15.8): compare an anomaly detector (Deep SVDD) built on a
   classification-pretrained embedding against one trained from scratch on
   clean sessions only, scored on an attack type held out entirely from
   all training.

Crucially, session generation below reuses this notebook's own already
-validated `channel_model`, `simulate_bb84_decoy`, `simulate_bkm07_pulse`,
and `run_e91` (Sections 1-3, 9, 13) directly -- there is no separate,
disconnected data-generation path to drift out of sync with the physics
audited earlier in this notebook.

**Runtime note:** the live run at the bottom of this section (15.9) uses
reduced `n_sessions_per_class`/epoch/fraction/seed counts so the whole
notebook still finishes in a few minutes -- it demonstrates the pipeline
and the qualitative result, not a publication-scale run. Scale those
numbers up (commented inline) for a real result with tight confidence
intervals.

In [ ]:
# ── DL imports (only needed from here on) ──────────────────────────────────
import copy as _copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, recall_score, roc_auc_score, confusion_matrix

DL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Deep-learning device: {DL_DEVICE}")

### 15.1 -- Attack taxonomy

Restricted to `eve_mode`s this notebook's own simulators actually
implement. An earlier draft of this section (see the audit chat history)
listed several attacks -- `beam_splitting`, `time_shift`,
`detector_blinding`, `source_flaw`, `trojan_horse`, `double_cnot` -- with
no real simulator behind them; fabricating labelled "attacks" as
made-up-shaped noise is exactly the kind of thing Section 13's note on the
four removed label-conditioned E91 features already rejected for the
classical pipeline, so the same standard applies here.

In [ ]:
DL_PROTOCOLS = ["bb84", "e91", "bkm07"]

DL_ATTACKS = {
    "bb84":  ["clean", "intercept_resend", "pns", "blocking", "loss_manipulation"],
    "e91":   ["clean", "intercept_resend", "ancilla", "loss_manipulation"],
    # simulate_bkm07_pulse has one interception mechanism; symmetric vs
    # asymmetric is a DATASET-level distinction (eve_fwd == eve_ret or not),
    # the same convention Section 4's generate_datasets() uses.
    "bkm07": ["clean", "symmetric_attack", "asymmetric_attack"],
}
DL_ATTACK_COUNTS = {p: len(v) for p, v in DL_ATTACKS.items()}

# Held out entirely from training for the zero-day test (Section 15.8) --
# chosen to be physically distinct from the rest of that protocol's list.
DL_ZERO_DAY_ATTACK_BY_TARGET = {
    "bb84":  "pns",                # near-zero QBER signature, distinct from intercept/loss attacks
    "e91":   "ancilla",            # basis-anisotropic dephasing, distinct from depolarising attacks
    "bkm07": "asymmetric_attack",  # distinct forward/return-leg signature from the symmetric case
}
print("DL attack taxonomy:", DL_ATTACKS)

### 15.2 -- Per-round feature encoding and windowing

Each round of any protocol is compressed into the same fixed-width, 8-slot
feature vector (`kept`, `error`, normalised `basis_a`/`basis_b`,
`basis_match`, `no_click`, `double_click`, a rescaled `chsh_running`), so
one shared trunk can read BB84, BKM07, and E91 rounds alike -- protocol
-specific physical meaning is handled upstream, in how each protocol's raw
simulator output gets mapped into these slots (Section 15.4). Sessions are
then sliced into fixed-length overlapping windows for the CNN/LSTM.

In [ ]:
N_FEATURES = 8


def encode_session(protocol, kept, error, basis_a, basis_b, no_click=None,
                    double_click=None, chsh_running=None, n_bases=2):
    """Assembles the (n_rounds, N_FEATURES) matrix for one session. All
    array arguments must be the same length (n_rounds,).

    n_bases: how many distinct values basis_a/basis_b take (2 for
    BB84/BKM07's Z/X, 3 for E91's three angle settings) -- used to
    normalise those two channels into [0, 1] so their scale is comparable
    across protocols despite differing basis-set sizes.
    """
    n = len(kept)
    kept = np.asarray(kept, dtype=np.float32)
    error = np.asarray(error, dtype=np.float32)
    basis_a = np.asarray(basis_a, dtype=np.float32)
    basis_b = np.asarray(basis_b, dtype=np.float32)
    no_click = np.zeros(n, dtype=np.float32) if no_click is None else np.asarray(no_click, dtype=np.float32)
    double_click = np.zeros(n, dtype=np.float32) if double_click is None else np.asarray(double_click, dtype=np.float32)
    chsh_running = np.zeros(n, dtype=np.float32) if chsh_running is None else np.asarray(chsh_running, dtype=np.float32)

    denom = max(n_bases - 1, 1)
    basis_a_n = basis_a / denom
    basis_b_n = basis_b / denom
    basis_match = (basis_a == basis_b).astype(np.float32)
    chsh_running_n = chsh_running / (2.0 * np.sqrt(2.0))   # rescale to O(1)

    X = np.stack([kept, error, basis_a_n, basis_b_n, basis_match,
                  no_click, double_click, chsh_running_n], axis=1)
    assert X.shape == (n, N_FEATURES)
    return X.astype(np.float32)


def make_windows(X, length=256, stride=128):
    """Overlapping fixed-length windows (n_windows, length, N_FEATURES).
    Sessions shorter than `length` are dropped -- zero-padding a short
    session would look like a genuine "everything quiet" window and bias
    the detectors, rather than a lack of data."""
    n = len(X)
    if n < length:
        return np.zeros((0, length, X.shape[1]), dtype=X.dtype)
    starts = range(0, n - length + 1, stride)
    return np.stack([X[s:s + length] for s in starts], axis=0)


print("encode_session() / make_windows() defined.")

### 15.3 -- Attention pooling and LSTM autoencoder

`AttentionPool` learns a scalar relevance score per timestep (softmax
-normalised over the window) instead of mean/last-hidden pooling, so the
network can emphasise whichever part of a window -- e.g. a burst -- carries
the attack signature. `LSTMAutoencoder` is an unsupervised
reconstruction-based building block (not used in the main flow below, kept
for anyone extending Section 15.8's ablation with a reconstruction-error
anomaly baseline alongside Deep SVDD).

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, hidden_dim, attn_dim=64):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim), nn.Tanh(), nn.Linear(attn_dim, 1))

    def forward(self, h):
        """h: (B, L, H) -> (pooled: (B, H), attn_weights: (B, L))"""
        logits = self.score(h).squeeze(-1)
        attn = torch.softmax(logits, dim=1)
        pooled = torch.einsum('bl,blh->bh', attn, h)
        return pooled, attn


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=N_FEATURES, hidden=64, latent=32, num_layers=1):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, num_layers=num_layers, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden, n_features)

    def forward(self, x):
        B, L, _ = x.shape
        _, (h_n, _) = self.encoder(x)
        z = self.to_latent(h_n[-1])
        dec_in = self.from_latent(z).unsqueeze(1).repeat(1, L, 1)
        dec_out, _ = self.decoder(dec_in)
        return self.out(dec_out), z

    def reconstruction_error(self, x):
        recon, _ = self.forward(x)
        return ((recon - x) ** 2).mean(dim=(1, 2))


print("AttentionPool / LSTMAutoencoder defined.")

### 15.4 -- Session generation, via this notebook's own physics

`simulate_session()` below calls `simulate_bb84_decoy` (Section 2),
`simulate_bkm07_pulse` (Section 2), and `run_e91` (Section 13) --
already-defined, already-validated in this same notebook -- rather than a
separate synthetic generator. `_rolling_chsh` recomputes E91's CHSH
statistic every `step` rounds from the trailing `window` rounds and holds
that value until the next recomputation (true per-round CHSH isn't
well-defined -- S needs many samples per angle-setting pair -- this is the
same "periodically report a running statistic" pattern a real experiment
would use).

**Performance note:** BKM07's simulator is a genuine per-pulse Python loop
(Section 2's note on why: round-trip loss makes most fired pulses "lost"
before a measurement, so vectorising wasn't worth it at this project's
scale) -- BKM07 sessions are the slow part of dataset generation below.

In [ ]:
from dataclasses import dataclass


@dataclass
class Session:
    protocol: str
    attack: str
    session_id: int
    X: np.ndarray


def _rolling_chsh(ak, bk, ra, rb, window=400, step=50):
    n = len(ak)
    chsh_running = np.zeros(n, dtype=np.float32)
    last_S = 0.0
    for end in range(step, n + step, step):
        end = min(end, n)
        start = max(0, end - window)
        corrs, ok = [], True
        for (a, b), s in zip(CHSH_PAIRS, CHSH_SIGNS):
            m = (ak[start:end] == a) & (bk[start:end] == b)
            if m.sum() < 5:
                ok = False
                break
            corrs.append(s * np.mean(ra[start:end][m] * rb[start:end][m]))
        if ok:
            last_S = float(sum(corrs))
        chsh_running[max(0, end - step):end] = last_S
    return chsh_running


def _bb84_dl_session(attack, strength, rng, n_rounds):
    distance_km = float(rng.uniform(*CHANNEL_DISTANCE_RANGE_KM))
    eve_mode = 'none' if attack == 'clean' else attack
    eve_intensity = 0.0 if attack == 'clean' else float(strength)
    profile = 'bursty' if (eve_mode == 'intercept_resend' and rng.random() < 0.5) else 'iid'

    run = simulate_bb84_decoy(n_rounds, distance_km, eve_mode=eve_mode,
                              eve_intensity=eve_intensity, profile=profile, rng=rng)
    kept = run['sift'].astype(np.float32)
    error = ((run['bit_A'] != run['bit_B']) & run['sift']).astype(np.float32)
    basis_a = run['bas_A'].astype(np.float32)
    basis_b = run['bas_B'].astype(np.float32)
    no_click = (~run['click']).astype(np.float32)
    double_click = np.zeros(n_rounds, dtype=np.float32)   # not modelled -- no per-detector click model
    return kept, error, basis_a, basis_b, no_click, double_click, None, 2


def _bkm07_dl_session(attack, strength, rng, n_rounds):
    distance_km = float(rng.uniform(*DISTANCE_RANGE_BKM))
    if attack == 'clean':
        eve_mode, eve_fwd, eve_ret = 'none', 0.0, 0.0
    elif attack == 'symmetric_attack':
        eve_mode, eve_fwd, eve_ret = 'symmetric', float(strength), float(strength)
    else:
        eve_mode = 'asymmetric'
        eve_fwd = float(strength * rng.uniform(0.1, 0.4))
        eve_ret = float(strength * rng.uniform(0.6, 1.0))

    kept = np.zeros(n_rounds, dtype=np.float32)
    error = np.zeros(n_rounds, dtype=np.float32)
    basis_a = np.zeros(n_rounds, dtype=np.float32)
    basis_b = np.zeros(n_rounds, dtype=np.float32)   # Bob's SIFT(0)/CTRL(1) mode
    no_click = np.zeros(n_rounds, dtype=np.float32)
    for i in range(n_rounds):
        p = simulate_bkm07_pulse(distance_km, eve_mode, eve_fwd, eve_ret, rng=rng)
        if p.get('lost', False):
            no_click[i] = 1.0
            continue
        basis_a[i] = p['basis_A']
        basis_b[i] = 0.0 if p['bob_mode'] == 'SIFT' else 1.0
        if p['round_type'] == 'SIFT_KEY':
            kept[i] = 1.0
            error[i] = float(p['bit_A'] != p['bit_A_final'])
    double_click = np.zeros(n_rounds, dtype=np.float32)
    return kept, error, basis_a, basis_b, no_click, double_click, None, 2


def _e91_dl_session(attack, strength, rng, n_rounds):
    V = sample_e91_channel(rng)
    eve_mode = 'none' if attack == 'clean' else attack
    eve_intensity = 0.0 if attack == 'clean' else float(strength)
    profile = 'bursty' if (eve_mode == 'intercept_resend' and rng.random() < 0.5) else 'iid'

    ak, bk, ra, rb = run_e91(n_rounds, V=V, eve_mode=eve_mode,
                             eve_intensity=eve_intensity, profile=profile, rng=rng)
    a_num = {name: i for i, name in enumerate(ALICE_ANGLES)}
    b_num = {name: i for i, name in enumerate(BOB_ANGLES)}
    basis_a = np.array([a_num[x] for x in ak], dtype=np.float32)
    basis_b = np.array([b_num[x] for x in bk], dtype=np.float32)

    key_codes = {a + b for a, b in KEY_PAIRS}
    pair_codes = np.char.add(ak, bk)
    kept = np.isin(pair_codes, list(key_codes)).astype(np.float32)
    error = (kept.astype(bool) & (ra == rb)).astype(np.float32)   # singlet anti-correlated

    chsh_running = _rolling_chsh(ak, bk, ra, rb)
    no_click = np.zeros(n_rounds, dtype=np.float32)     # E91's CPTP channel has no loss concept
    double_click = np.zeros(n_rounds, dtype=np.float32)
    return kept, error, basis_a, basis_b, no_click, double_click, chsh_running, 3


_DL_SESSION_BUILDERS = {"bb84": _bb84_dl_session, "e91": _e91_dl_session, "bkm07": _bkm07_dl_session}


def simulate_session(protocol, attack, session_id, rng, n_rounds=4000):
    strength = float(rng.uniform(0.05, 1.0))
    kept, error, basis_a, basis_b, no_click, double_click, chsh_running, n_bases = \
        _DL_SESSION_BUILDERS[protocol](attack, strength, rng, n_rounds)
    X = encode_session(protocol, kept, error, basis_a, basis_b, no_click=no_click,
                       double_click=double_click, chsh_running=chsh_running, n_bases=n_bases)
    return Session(protocol=protocol, attack=attack, session_id=session_id, X=X)


def build_dl_dataset(n_sessions_per_class=40, seed=0, window=256, stride=128, n_rounds=4000):
    rng = np.random.default_rng(seed)
    Xs, protos, attacks_fine, is_attacked, groups = [], [], [], [], []
    sid = 0
    for proto in DL_PROTOCOLS:
        for a_idx, attack in enumerate(DL_ATTACKS[proto]):
            for _ in range(n_sessions_per_class):
                s = simulate_session(proto, attack, sid, rng, n_rounds=n_rounds)
                win = make_windows(s.X, length=window, stride=stride)
                Xs.append(win)
                protos += [proto] * len(win)
                attacks_fine += [a_idx] * len(win)
                is_attacked += [0.0 if attack == 'clean' else 1.0] * len(win)
                groups += [sid] * len(win)
                sid += 1
    X = np.concatenate(Xs, axis=0).astype(np.float32)
    return {"X": X, "protocol": np.array(protos),
            "attack_fine": np.array(attacks_fine, dtype=np.int64),
            "is_attacked": np.array(is_attacked, dtype=np.float32),
            "group": np.array(groups, dtype=np.int64)}


def session_split(groups, test_size=0.2, val_size=0.1, seed=0):
    """Splits by SESSION ID, never by window -- required to avoid leakage
    (the same window-vs-session-split discipline as Section 4's train/test
    split, just at the level of individual round-windows here)."""
    idx = np.arange(len(groups))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval_idx, groups=groups[trainval_idx]))
    return trainval_idx[train_idx], trainval_idx[val_idx], test_idx


print("Session generation (simulate_session / build_dl_dataset / session_split) defined.")

### 15.5 -- CrossProtocolDetector: shared trunk, per-protocol adapters

A linear **adapter** per protocol projects each protocol's 8-feature
window into a shared latent space; a dilated Conv1d stack + bidirectional
LSTM + attention pooling (the shared **trunk**) then extracts one
embedding per window. Two kinds of head read that embedding: a **shared**
binary head ("attacked or not" -- the only label meaning the same thing in
every protocol, and the transfer target for Section 15.7), and
**per-protocol** multiclass heads ("which attack" -- NOT shared, since the
three protocols' attack lists don't overlap).

In [ ]:
class CrossProtocolDetector(nn.Module):
    def __init__(self, protocols=DL_PROTOCOLS, attack_counts=DL_ATTACK_COUNTS,
                n_features=N_FEATURES, latent=48, hidden=64, dropout=0.2):
        super().__init__()
        self.protocols = protocols
        self.adapters = nn.ModuleDict({p: nn.Linear(n_features, latent) for p in protocols})
        self.conv = nn.Sequential(
            nn.Conv1d(latent, hidden, 5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, 5, padding=4, dilation=2), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, 3, padding=4, dilation=4), nn.BatchNorm1d(hidden), nn.ReLU(),
        )
        self.lstm = nn.LSTM(hidden, hidden, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.pool = AttentionPool(2 * hidden)
        self.binary_head = nn.Linear(2 * hidden, 1)                                    # shared
        self.fine_heads = nn.ModuleDict({p: nn.Linear(2 * hidden, attack_counts[p]) for p in protocols})  # not shared

    def encode(self, x, proto_list):
        z = torch.stack([self.adapters[p](xi) for xi, p in zip(x, proto_list)])
        h = self.conv(z.transpose(1, 2)).transpose(1, 2)
        h, _ = self.lstm(h)
        return self.pool(h)

    def forward_binary(self, x, proto_list):
        pooled, attn = self.encode(x, proto_list)
        return self.binary_head(pooled).squeeze(-1), attn

    def forward_fine(self, x, proto_list):
        assert len(set(proto_list)) == 1, "forward_fine expects a single-protocol batch"
        pooled, attn = self.encode(x, proto_list)
        return self.fine_heads[proto_list[0]](pooled), attn

    def freeze_trunk(self):
        for module in (self.conv, self.lstm, self.pool):
            for p in module.parameters():
                p.requires_grad = False

    def freeze_adapter(self, protocol):
        for p in self.adapters[protocol].parameters():
            p.requires_grad = False


print("CrossProtocolDetector defined.")

### 15.6 -- Training and evaluation: binary and per-protocol fine heads

In [ ]:
def train_binary(model, train_loader, val_loader, device, epochs=25, lr=1e-3, verbose=True):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    best_f1, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        for xb, pb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model.forward_binary(xb, list(pb))
            loss = F.binary_cross_entropy_with_logits(logits, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        sched.step()
        metrics = evaluate_binary(model, val_loader, device)
        if verbose:
            print(f"  epoch {ep+1:3d}  val_F1 {metrics['f1']:.4f}  val_recall {metrics['recall']:.4f}")
        if metrics["f1"] > best_f1:
            best_f1, best_state = metrics["f1"], _copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def evaluate_binary(model, loader, device):
    model.eval()
    all_true, all_prob = [], []
    for xb, pb, yb in loader:
        logits, _ = model.forward_binary(xb.to(device), list(pb))
        all_prob.append(torch.sigmoid(logits).cpu().numpy())
        all_true.append(yb.numpy())
    y_true, y_prob = np.concatenate(all_true), np.concatenate(all_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    out = {"f1": f1_score(y_true, y_pred, zero_division=0),
          "recall": recall_score(y_true, y_pred, zero_division=0)}
    try:
        out["auc"] = roc_auc_score(y_true, y_prob)
    except ValueError:
        out["auc"] = float("nan")
    return out


def train_fine(model, protocol, train_loader, val_loader, device, epochs=25, lr=1e-3, verbose=True):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    best_f1, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        for xb, pb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model.forward_fine(xb, list(pb))
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        macro_f1, _ = evaluate_fine(model, protocol, val_loader, device)
        if verbose:
            print(f"  [{protocol}] epoch {ep+1:3d}  val_macroF1 {macro_f1:.4f}")
        if macro_f1 > best_f1:
            best_f1, best_state = macro_f1, _copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def evaluate_fine(model, protocol, loader, device):
    model.eval()
    all_true, all_pred = [], []
    for xb, pb, yb in loader:
        logits, _ = model.forward_fine(xb.to(device), list(pb))
        all_pred.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
    y_true, y_pred = np.concatenate(all_true), np.concatenate(all_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(DL_ATTACKS[protocol]))))
    return macro_f1, cm


print("train_binary / evaluate_binary / train_fine / evaluate_fine defined.")

### 15.7 -- Leave-one-protocol-out transfer (headline experiment)

For each protocol as target in turn: pretrain the shared trunk + the
*other two* protocols' adapters jointly (binary head only); freeze the
trunk; fine-tune only the target's (still randomly-initialised) adapter on
a small fraction of target data; compare against a from-scratch model
trained on that same fraction; evaluate both on the full held-out target
test set, across fractions and seeds. If transfer beats scratch especially
at low fractions, in most/all three directions, the trunk has learned
attack signatures that generalise across protocols.

In [ ]:
def make_dl_loader(X, protocol_arr, y, idx, batch_size=64, shuffle=True):
    class _DS(Dataset):
        def __len__(self):
            return len(idx)

        def __getitem__(self, i):
            j = idx[i]
            return X[j], protocol_arr[j], y[j]

    def collate(batch):
        xs = torch.from_numpy(np.stack([b[0] for b in batch])).float()
        ps = [b[1] for b in batch]
        ys = torch.from_numpy(np.array([b[2] for b in batch], dtype=np.float32))
        return xs, ps, ys

    return DataLoader(_DS(), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


def transfer_vs_scratch(data, device, source_protocols, target_protocol,
                        fractions=(0.05, 0.1, 0.25, 0.5, 1.0), seeds=(0, 1, 2),
                        epochs_pretrain=20, epochs_finetune=15):
    X, proto, y, groups = data["X"], data["protocol"], data["is_attacked"], data["group"]
    results = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        torch.manual_seed(seed)

        src_mask = np.isin(proto, source_protocols)
        src_train, src_val, _ = session_split(groups[src_mask], seed=seed)
        src_idx_all = np.flatnonzero(src_mask)
        src_train_idx, src_val_idx = src_idx_all[src_train], src_idx_all[src_val]

        tgt_mask = proto == target_protocol
        tgt_idx_all = np.flatnonzero(tgt_mask)
        tgt_groups = groups[tgt_idx_all]
        tgt_train_pool_rel, tgt_val_rel, tgt_test_rel = session_split(tgt_groups, seed=seed)
        tgt_train_pool = tgt_idx_all[tgt_train_pool_rel]
        tgt_val_idx = tgt_idx_all[tgt_val_rel]
        tgt_test_idx = tgt_idx_all[tgt_test_rel]

        src_train_loader = make_dl_loader(X, proto, y, src_train_idx)
        src_val_loader = make_dl_loader(X, proto, y, src_val_idx, shuffle=False)
        tgt_val_loader = make_dl_loader(X, proto, y, tgt_val_idx, shuffle=False)
        tgt_test_loader = make_dl_loader(X, proto, y, tgt_test_idx, shuffle=False)

        print(f"\n[{'+'.join(source_protocols)} -> {target_protocol}] seed {seed}: pretraining shared trunk ...")
        pretrained = CrossProtocolDetector()
        pretrained = train_binary(pretrained, src_train_loader, src_val_loader, device,
                                  epochs=epochs_pretrain, verbose=False)

        for frac in fractions:
            uniq_sessions = np.unique(groups[tgt_train_pool])
            n_take = max(1, int(len(uniq_sessions) * frac))
            take_sessions = set(rng.choice(uniq_sessions, size=n_take, replace=False))
            frac_idx = tgt_train_pool[np.isin(groups[tgt_train_pool], list(take_sessions))]
            frac_train_loader = make_dl_loader(X, proto, y, frac_idx)

            transfer_model = _copy.deepcopy(pretrained)
            transfer_model.freeze_trunk()
            for sp in source_protocols:
                transfer_model.freeze_adapter(sp)
            transfer_model = train_binary(transfer_model, frac_train_loader, tgt_val_loader,
                                          device, epochs=epochs_finetune, verbose=False)
            transfer_metrics = evaluate_binary(transfer_model, tgt_test_loader, device)

            scratch_model = CrossProtocolDetector()
            scratch_model = train_binary(scratch_model, frac_train_loader, tgt_val_loader,
                                         device, epochs=epochs_finetune, verbose=False)
            scratch_metrics = evaluate_binary(scratch_model, tgt_test_loader, device)

            results.append({
                "target_protocol": target_protocol, "source_protocols": tuple(source_protocols),
                "seed": seed, "fraction": frac, "n_sessions": n_take,
                "transfer_f1": transfer_metrics["f1"], "transfer_recall": transfer_metrics["recall"],
                "transfer_auc": transfer_metrics["auc"],
                "scratch_f1": scratch_metrics["f1"], "scratch_recall": scratch_metrics["recall"],
                "scratch_auc": scratch_metrics["auc"],
            })
            print(f"  frac={frac:<5} n_sessions={n_take:<4} "
                 f"transfer F1={transfer_metrics['f1']:.3f} AUC={transfer_metrics['auc']:.3f}  |  "
                 f"scratch F1={scratch_metrics['f1']:.3f} AUC={scratch_metrics['auc']:.3f}")
    return results


def run_leave_one_protocol_out(data, device, fractions=(0.1, 0.25, 0.5, 1.0), seeds=(0, 1),
                               epochs_pretrain=20, epochs_finetune=15):
    all_results = {}
    for target in DL_PROTOCOLS:
        sources = [p for p in DL_PROTOCOLS if p != target]
        all_results[target] = transfer_vs_scratch(
            data, device, source_protocols=sources, target_protocol=target,
            fractions=fractions, seeds=seeds,
            epochs_pretrain=epochs_pretrain, epochs_finetune=epochs_finetune)
    return all_results


def summarize_dl_results(results, title=None):
    by_frac = {}
    for r in results:
        by_frac.setdefault(r["fraction"], []).append(r)
    if title:
        print(f"\n### {title} ###")
    print(f"\n{'fraction':>8} {'n_ses':>6} {'transfer_F1':>14} {'scratch_F1':>13} {'transfer_AUC':>14} {'scratch_AUC':>13}")
    for frac, rows in sorted(by_frac.items()):
        tf1 = np.array([r["transfer_f1"] for r in rows]); sf1 = np.array([r["scratch_f1"] for r in rows])
        tauc = np.array([r["transfer_auc"] for r in rows]); sauc = np.array([r["scratch_auc"] for r in rows])
        print(f"{frac:>8.2f} {rows[0]['n_sessions']:>6} "
             f"{tf1.mean():>7.3f}+-{tf1.std():<5.3f} {sf1.mean():>6.3f}+-{sf1.std():<5.3f} "
             f"{tauc.mean():>7.3f}+-{tauc.std():<5.3f} {sauc.mean():>6.3f}+-{sauc.std():<5.3f}")


def plot_dl_data_efficiency(results, metric="f1", save_path=None, title=None):
    target = results[0]["target_protocol"]; sources = results[0]["source_protocols"]
    save_path = save_path or f"plots/dl_transfer_{'_'.join(sources)}_to_{target}.png"
    title = title or f"Cross-protocol transfer: {'+'.join(sources)} -> {target}"
    by_frac = {}
    for r in results:
        by_frac.setdefault(r["fraction"], []).append(r)
    fracs = sorted(by_frac.keys())
    t_mean = np.array([np.mean([r[f"transfer_{metric}"] for r in by_frac[f]]) for f in fracs])
    t_std = np.array([np.std([r[f"transfer_{metric}"] for r in by_frac[f]]) for f in fracs])
    s_mean = np.array([np.mean([r[f"scratch_{metric}"] for r in by_frac[f]]) for f in fracs])
    s_std = np.array([np.std([r[f"scratch_{metric}"] for r in by_frac[f]]) for f in fracs])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fracs, t_mean, "o-", color="#2166ac", label=f"Transfer (pretrained {'+'.join(sources)})")
    ax.fill_between(fracs, t_mean - t_std, t_mean + t_std, color="#2166ac", alpha=0.15)
    ax.plot(fracs, s_mean, "s--", color="#b2182b", label=f"Scratch ({target} only)")
    ax.fill_between(fracs, s_mean - s_std, s_mean + s_std, color="#b2182b", alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel(f"Fraction of {target} training sessions used")
    ax.set_ylabel(f"{metric.upper()} on held-out {target} test set")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=150)
    print(f"saved plot to {save_path}")
    return fig


print("Leave-one-protocol-out transfer machinery defined.")

### 15.8 -- Deep SVDD anomaly branch: does supervised pretraining help zero-day detection?

Deep SVDD, in one paragraph: pick a fixed center `c` in embedding space,
train the network so normal (clean) sessions' embeddings land close to
`c`; at test time, distance-from-`c` is the anomaly score. `c` is fixed
(not learned) to stop the trivial collapse where the network could
otherwise map everything to `c` and "cheat".

**Branch A** (supervised representation): take a model already trained
with the binary classification loss (Section 15.7's transfer model),
freeze it completely, fit only the center statistically. **Branch B**
(normal-only representation): a fresh model, trunk trained purely with the
SVDD loss on clean sessions -- classification labels never used. Both
scored by ROC-AUC on a **zero-day holdout**: an attack type excluded
entirely from all training (Section 15.1's `DL_ZERO_DAY_ATTACK_BY_TARGET`),
across all three leave-one-out directions.

In [ ]:
@torch.no_grad()
def compute_center(model, loader, device, eps=0.1):
    model.eval()
    embs = []
    for xb, pb, yb in loader:
        mask = yb.numpy() == 0.0
        if mask.sum() == 0:
            continue
        pooled, _ = model.encode(xb[mask].to(device), [p for p, m in zip(pb, mask) if m])
        embs.append(pooled.cpu().numpy())
    c = np.concatenate(embs, axis=0).mean(axis=0)
    c[(np.abs(c) < eps) & (c >= 0)] = eps
    c[(np.abs(c) < eps) & (c < 0)] = -eps
    return torch.tensor(c, dtype=torch.float32, device=device)


def train_svdd(model, clean_train_loader, clean_val_loader, device, epochs=25, lr=1e-3, verbose=True):
    model.to(device)
    center = compute_center(model, clean_train_loader, device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)
    best_loss, best_state = float("inf"), None
    for ep in range(epochs):
        model.train()
        for xb, pb, yb in clean_train_loader:
            mask = yb.numpy() == 0.0
            if mask.sum() == 0:
                continue
            pooled, _ = model.encode(xb[mask].to(device), [p for p, m in zip(pb, mask) if m])
            loss = ((pooled - center) ** 2).sum(dim=1).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        val_loss = svdd_loss(model, center, clean_val_loader, device)
        if verbose:
            print(f"  SVDD epoch {ep+1:3d}  val_dist {val_loss:.4f}")
        if val_loss < best_loss:
            best_loss, best_state = val_loss, _copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, center


@torch.no_grad()
def svdd_loss(model, center, loader, device):
    model.eval()
    dists = []
    for xb, pb, yb in loader:
        mask = yb.numpy() == 0.0
        if mask.sum() == 0:
            continue
        pooled, _ = model.encode(xb[mask].to(device), [p for p, m in zip(pb, mask) if m])
        dists.append(((pooled - center) ** 2).sum(dim=1).cpu().numpy())
    return float(np.concatenate(dists).mean()) if dists else float("inf")


@torch.no_grad()
def anomaly_scores(model, center, loader, device):
    model.eval()
    scores, labels = [], []
    for xb, pb, yb in loader:
        pooled, _ = model.encode(xb.to(device), list(pb))
        scores.append(((pooled - center) ** 2).sum(dim=1).cpu().numpy())
        labels.append(yb.numpy())
    return np.concatenate(scores), np.concatenate(labels)


def evaluate_anomaly(model, center, loader, device):
    scores, labels = anomaly_scores(model, center, loader, device)
    try:
        auc = roc_auc_score(labels, scores)
    except ValueError:
        auc = float("nan")
    return {"auc": auc, "scores": scores, "labels": labels}


def make_zero_day_loader(data, target_protocol, held_out_attack, idx, batch_size=64):
    """Restricts idx to {clean, held_out_attack} windows, relabeled as a
    binary anomaly problem. NOTE: the relabeled array is built at full
    length (len(data["X"])), not len(idx) -- make_dl_loader indexes it
    globally."""
    X, proto, attack_fine = data["X"], data["protocol"], data["attack_fine"]
    held_idx = DL_ATTACKS[target_protocol].index(held_out_attack)
    clean_idx_local = DL_ATTACKS[target_protocol].index("clean")
    keep = np.isin(attack_fine[idx], [clean_idx_local, held_idx])
    sub = idx[keep]
    y_zd = np.zeros(len(X), dtype=np.float32)
    y_zd[sub] = (attack_fine[sub] == held_idx).astype(np.float32)
    return make_dl_loader(X, proto, y_zd, sub, batch_size=batch_size, shuffle=False)


def run_representation_ablation(data, device, target_protocol, source_protocols,
                                fractions=(0.25, 1.0), seeds=(0, 1), held_out_attack=None,
                                epochs_pretrain=20, epochs_svdd=25):
    X, proto, y, groups = data["X"], data["protocol"], data["is_attacked"], data["group"]
    rows = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        torch.manual_seed(seed)

        src_mask = np.isin(proto, source_protocols)
        src_train, src_val, _ = session_split(groups[src_mask], seed=seed)
        src_idx_all = np.flatnonzero(src_mask)
        src_train_loader = make_dl_loader(X, proto, y, src_idx_all[src_train])
        src_val_loader = make_dl_loader(X, proto, y, src_idx_all[src_val], shuffle=False)

        tgt_mask = proto == target_protocol
        tgt_idx_all = np.flatnonzero(tgt_mask)
        tgt_groups = groups[tgt_idx_all]
        tgt_train_pool_rel, tgt_val_rel, tgt_test_rel = session_split(tgt_groups, seed=seed)
        tgt_train_pool = tgt_idx_all[tgt_train_pool_rel]
        tgt_val_idx = tgt_idx_all[tgt_val_rel]
        tgt_test_idx = tgt_idx_all[tgt_test_rel]

        tgt_test_loader = (make_zero_day_loader(data, target_protocol, held_out_attack, tgt_test_idx)
                           if held_out_attack else make_dl_loader(X, proto, y, tgt_test_idx, shuffle=False))

        supervised_trunk = CrossProtocolDetector()
        supervised_trunk = train_binary(supervised_trunk, src_train_loader, src_val_loader,
                                        device, epochs=epochs_pretrain, verbose=False)

        for frac in fractions:
            uniq_sessions = np.unique(groups[tgt_train_pool])
            n_take = max(1, int(len(uniq_sessions) * frac))
            take_sessions = set(rng.choice(uniq_sessions, size=n_take, replace=False))
            frac_idx = tgt_train_pool[np.isin(groups[tgt_train_pool], list(take_sessions))]
            frac_train_loader = make_dl_loader(X, proto, y, frac_idx)
            tgt_val_loader = make_dl_loader(X, proto, y, tgt_val_idx, shuffle=False)

            model_a = _copy.deepcopy(supervised_trunk)
            model_a.freeze_trunk()
            for sp in source_protocols:
                model_a.freeze_adapter(sp)
            model_a = train_binary(model_a, frac_train_loader, tgt_val_loader, device, epochs=10, verbose=False)
            for p in model_a.parameters():
                p.requires_grad = False
            center_a = compute_center(model_a, frac_train_loader, device)
            metrics_a = evaluate_anomaly(model_a, center_a, tgt_test_loader, device)

            model_b = CrossProtocolDetector()
            model_b, center_b = train_svdd(model_b, frac_train_loader, tgt_val_loader, device,
                                           epochs=epochs_svdd, verbose=False)
            metrics_b = evaluate_anomaly(model_b, center_b, tgt_test_loader, device)

            rows.append({"target_protocol": target_protocol, "seed": seed, "fraction": frac,
                        "held_out_attack": held_out_attack,
                        "supervised_repr_auc": metrics_a["auc"], "normal_only_repr_auc": metrics_b["auc"]})
            print(f"[{target_protocol}] frac={frac} seed={seed} "
                 f"supervised-repr AUC={metrics_a['auc']:.3f}  normal-only-repr AUC={metrics_b['auc']:.3f}")
    return rows


def summarize_ablation(rows):
    by_frac = {}
    for r in rows:
        by_frac.setdefault(r["fraction"], []).append(r)
    print(f"\n{'fraction':>8} {'supervised_repr_AUC':>20} {'normal_only_repr_AUC':>22}")
    for frac, rs in sorted(by_frac.items()):
        a = np.array([r["supervised_repr_auc"] for r in rs]); b = np.array([r["normal_only_repr_auc"] for r in rs])
        print(f"{frac:>8.2f} {a.mean():>14.3f}+-{a.std():<5.3f} {b.mean():>16.3f}+-{b.std():<5.3f}")


def run_representation_ablation_all_directions(data, device, fractions=(0.25, 1.0), seeds=(0, 1),
                                               held_out_by_target=None, epochs_pretrain=20, epochs_svdd=25):
    held_out_by_target = held_out_by_target or DL_ZERO_DAY_ATTACK_BY_TARGET
    all_rows = []
    for target in DL_PROTOCOLS:
        sources = [p for p in DL_PROTOCOLS if p != target]
        held_out = held_out_by_target[target]
        print(f"\n--- Ablation direction: {'+'.join(sources)} -> {target} (zero-day attack: {held_out}) ---")
        all_rows.extend(run_representation_ablation(
            data, device, target_protocol=target, source_protocols=sources,
            fractions=fractions, seeds=seeds, held_out_attack=held_out,
            epochs_pretrain=epochs_pretrain, epochs_svdd=epochs_svdd))
    return all_rows


def summarize_ablation_all(all_rows):
    by_target = {}
    for r in all_rows:
        by_target.setdefault(r["target_protocol"], []).append(r)
    for target, rows in by_target.items():
        print(f"\n### Direction: -> {target}  (zero-day attack: {rows[0]['held_out_attack']}) ###")
        summarize_ablation(rows)
    sup_wins = sum(1 for r in all_rows if r["supervised_repr_auc"] > r["normal_only_repr_auc"])
    norm_wins = sum(1 for r in all_rows if r["normal_only_repr_auc"] > r["supervised_repr_auc"])
    ties = len(all_rows) - sup_wins - norm_wins
    avg_sup = np.nanmean([r["supervised_repr_auc"] for r in all_rows])
    avg_norm = np.nanmean([r["normal_only_repr_auc"] for r in all_rows])
    print(f"\n=== OVERALL, across all {len(all_rows)} (direction x fraction x seed) runs ===")
    print(f"supervised-representation wins: {sup_wins}   normal-only-representation wins: {norm_wins}   ties: {ties}")
    print(f"mean AUC -- supervised: {avg_sup:.3f}   normal-only: {avg_norm:.3f}")
    if avg_sup > avg_norm:
        print("On average, cross-protocol attack-labeled pretraining HELPS zero-day detection.")
    else:
        print("On average, cross-protocol attack-labeled pretraining HURTS zero-day detection")
        print("relative to a representation learned from normal behavior alone.")


print("Deep SVDD anomaly branch defined.")

### 15.9 -- Running it

Reduced `n_sessions_per_class`/epoch/fraction/seed counts below keep this
runnable as part of the notebook's normal top-to-bottom pass (a few
minutes, not the tens of minutes a publication-scale sweep would take).
Scale up the commented-out "real run" values for a result with tight
confidence intervals -- the machinery itself doesn't change.

In [ ]:
# Reduced for notebook runtime. For a real run: n_sessions_per_class=40-60,
# epochs_pretrain=20, epochs_finetune=15, fractions=(0.05,0.1,0.25,0.5,1.0),
# seeds=(0,1,2) -- same machinery, just more of it.
print("Building the DL dataset from this notebook's own physics simulators...")
dl_data = build_dl_dataset(n_sessions_per_class=12, window=192, stride=96, n_rounds=2000)
print(f"total windows: {len(dl_data['X'])}   feature dim: {dl_data['X'].shape[-1]}")
for p in DL_PROTOCOLS:
    n = (dl_data["protocol"] == p).sum()
    print(f"  {p:6s}: {n} windows across {len(DL_ATTACKS[p])} classes {DL_ATTACKS[p]}")

In [ ]:
print("\n=== Per-protocol fine attack-type classifier ===")
for p in DL_PROTOCOLS:
    mask = dl_data["protocol"] == p
    idx_all = np.flatnonzero(mask)
    tr, va, te = session_split(dl_data["group"][mask], seed=0)
    tr_idx, va_idx, te_idx = idx_all[tr], idx_all[va], idx_all[te]

    def _loader_fine(idx, shuffle):
        class _DS(Dataset):
            def __len__(self): return len(idx)
            def __getitem__(self, i):
                j = idx[i]
                return dl_data["X"][j], dl_data["protocol"][j], dl_data["attack_fine"][j]
        def _collate(batch):
            xs = torch.from_numpy(np.stack([b[0] for b in batch])).float()
            ps = [b[1] for b in batch]
            ys = torch.tensor([b[2] for b in batch], dtype=torch.long)
            return xs, ps, ys
        return DataLoader(_DS(), batch_size=64, shuffle=shuffle, collate_fn=_collate)

    model_p = CrossProtocolDetector()
    model_p = train_fine(model_p, p, _loader_fine(tr_idx, True), _loader_fine(va_idx, False),
                         DL_DEVICE, epochs=6, verbose=False)
    macro_f1, cm = evaluate_fine(model_p, p, _loader_fine(te_idx, False), DL_DEVICE)
    print(f"[{p}] TEST macro-F1: {macro_f1:.4f}")
    print(f"[{p}] confusion matrix (rows=true, cols=pred, order={DL_ATTACKS[p]}):\n{cm}")

In [ ]:
print("\n=== HEADLINE EXPERIMENT: leave-one-protocol-out transfer, all 3 directions ===")
dl_loo_results = run_leave_one_protocol_out(dl_data, DL_DEVICE, fractions=(0.25, 1.0),
                                            seeds=(0,), epochs_pretrain=6, epochs_finetune=5)
for target, results in dl_loo_results.items():
    sources = results[0]["source_protocols"]
    summarize_dl_results(results, title=f"{'+'.join(sources)} -> {target}")
    plot_dl_data_efficiency(results, metric="f1")
print("\nIf transfer_F1 > scratch_F1 especially at LOW fractions, in most/all three")
print("directions, that is the headline result: the shared trunk learns attack")
print("signatures that generalize ACROSS protocols, not just for one convenient pairing.")
print("(Single seed / reduced epochs here -- widen seeds for a real confidence interval.)")

In [ ]:
print("\n=== SECONDARY EXPERIMENT: supervised vs normal-only representation for anomaly detection ===")
for t in DL_PROTOCOLS:
    print(f"  -> {t}: holding out '{DL_ZERO_DAY_ATTACK_BY_TARGET[t]}' entirely from training")

dl_ablation_rows = run_representation_ablation_all_directions(
    dl_data, DL_DEVICE, fractions=(1.0,), seeds=(0,), epochs_pretrain=6, epochs_svdd=6)
summarize_ablation_all(dl_ablation_rows)
plot_ablation_all_directions_path = "plots/dl_representation_ablation_all_directions.png"


def plot_ablation_all_directions(all_rows, save_path=plot_ablation_all_directions_path):
    fracs = sorted(set(r["fraction"] for r in all_rows))
    targets = DL_PROTOCOLS
    fig, axes = plt.subplots(1, len(fracs), figsize=(5 * len(fracs), 4.5), sharey=True)
    if len(fracs) == 1:
        axes = [axes]
    width = 0.35
    x = np.arange(len(targets))
    for ax, frac in zip(axes, fracs):
        sup_means, sup_stds, norm_means, norm_stds = [], [], [], []
        for t in targets:
            rs = [r for r in all_rows if r["target_protocol"] == t and r["fraction"] == frac]
            sup = np.array([r["supervised_repr_auc"] for r in rs]); norm = np.array([r["normal_only_repr_auc"] for r in rs])
            sup_means.append(np.nanmean(sup)); sup_stds.append(np.nanstd(sup))
            norm_means.append(np.nanmean(norm)); norm_stds.append(np.nanstd(norm))
        ax.bar(x - width / 2, sup_means, width, yerr=sup_stds, capsize=4, color="#2166ac", label="Supervised representation")
        ax.bar(x + width / 2, norm_means, width, yerr=norm_stds, capsize=4, color="#b2182b", label="Normal-only representation")
        ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels([f"-> {t}\n({DL_ZERO_DAY_ATTACK_BY_TARGET[t]})" for t in targets])
        ax.set_title(f"fraction = {frac}"); ax.set_ylim(0.0, 1.0)
    axes[0].set_ylabel("Zero-day detection ROC-AUC")
    axes[-1].legend(loc="lower right")
    fig.suptitle("Supervised vs normal-only representation, all 3 leave-one-out directions")
    fig.tight_layout(); fig.savefig(save_path, dpi=150)
    print(f"saved plot to {save_path}")
    return fig


plot_ablation_all_directions(dl_ablation_rows)